# Workstream 1 - RHNA and Housing Production

This notebook prepares and validates the data needed for two separate dashboard tabs:

1. **RHNA**
2. **Housing Production**

Coverage:
- HCD APR annual activity: **2018-2025**
- HCD 6th Cycle RHNA allocation/progress: cumulative current-cycle snapshot
- Housing production by HCD housing type: **2018-2025**
- California DOF E-5 annual housing stock: **2020-2025**
- Housing-stock benchmarks: **2000 Census, 2010 Census, 2021 ACS 5-year, 2024 ACS 5-year**

Development stages remain separate throughout the notebook:
applications, entitlements, building permits, and completions are not combined.
Where HCD APR does not provide a reliable regionwide measure (for example,
units under construction), the limitation is documented rather than inferred.


## Setup

In [207]:
%pip install pandas numpy requests openpyxl

Note: you may need to restart the kernel to use updated packages.


In [208]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").exists():
            return candidate
        my_folder = candidate / "van's work"
        if (my_folder / "notebooks").exists():
            return my_folder
    raise FileNotFoundError(
        "Could not locate workstream root (no notebooks/ folder found nearby)."
    )


ROOT = find_workstream_root()
RAW_DIR = ROOT / "data" / "raw" / "hcd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

from datetime import date
DOWNLOAD_DATE = date.today().isoformat()

print("Workstream root:", ROOT)
print("Download date:", DOWNLOAD_DATE)


Workstream root: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work
Download date: 2026-08-17


In [209]:
TARGET_YEAR = 2025
ACS_VINTAGE_LABEL = "2020-2024"
ACS_DATA_YEAR = 2024

SAN_DIEGO_CITIES = [
    "Carlsbad", "Chula Vista", "Coronado", "Del Mar", "El Cajon",
    "Encinitas", "Escondido", "Imperial Beach", "La Mesa", "Lemon Grove",
    "National City", "Oceanside", "Poway", "San Diego", "San Marcos",
    "Santee", "Solana Beach", "Vista",
]
COUNTY_JURISDICTION_NAME = "Unincorporated San Diego County"


In [210]:
COUNTY_NAME_VARIANTS = {
    "san diego county", "county of san diego", "s d county",
    "county san diego", "unincorporated", "unincorporated san diego county",
}


def normalize_jurisdiction(name: object) -> str:
    s = str(name).strip().lower()
    if s == "national city":
        return "national city"
    if s in COUNTY_NAME_VARIANTS:
        return "unincorporated san diego county"
    s = re.sub(r"^city\s+of\s+", "", s)
    s = re.sub(r"^county\s+of\s+", "county ", s)
    s = re.sub(r"\s+city$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


sd_jur_keys = {normalize_jurisdiction(c) for c in SAN_DIEGO_CITIES}
sd_jur_keys.add(normalize_jurisdiction(COUNTY_JURISDICTION_NAME))


def get_json(url: str, params: dict | None = None, timeout: int = 60):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "CHPD Housing Dashboard Data Validation"},
    )
    if not response.ok:
        raise RuntimeError(f"Request failed ({response.status_code}): {response.url}")
    return response.json()


def find_resource_download_url(package_json: dict, name_contains: str):
    resources = package_json["result"]["resources"]
    matches = [
        r for r in resources
        if name_contains.lower() in r.get("name", "").lower()
        and r.get("format", "").upper() == "CSV"
    ]
    if not matches:
        raise ValueError(f"No CSV resource matching '{name_contains}' found.")
    matches.sort(key=lambda r: r.get("last_modified", ""), reverse=True)
    return matches[0]["url"], matches[0]["name"]


def fill_missing_jurisdiction_years(
    df: pd.DataFrame,
    years: list,
) -> pd.DataFrame:
    """
    Guarantee every jurisdiction-year combination exists without
    converting missing source records into zero.

    Existing source rows keep their reported values, including genuine
    reported zeros. Jurisdiction-years absent from the source remain NaN
    and are labeled 'missing from source'.
    """

    full_index = pd.MultiIndex.from_product(
        [
            sorted(sd_jur_keys),
            years,
        ],
        names=[
            "jur_clean",
            "year",
        ],
    )

    working = df.copy()

    working["_source_row_present"] = True

    filled = (
        working
        .set_index(
            [
                "jur_clean",
                "year",
            ]
        )
        .reindex(full_index)
        .reset_index()
    )

    filled["reporting_status"] = np.where(
        filled[
            "_source_row_present"
        ].fillna(False),
        "reported",
        "missing from source",
    )

    filled = filled.drop(
        columns=[
            "_source_row_present"
        ]
    )

    return filled

## APR (permits, entitlements, completions)

In [211]:
CKAN_PACKAGE_URL = (
    "https://data.ca.gov/api/3/action/package_show"
    "?id=housing-element-annual-progress-report-apr-data-by-jurisdiction-and-year"
)
package_json = get_json(CKAN_PACKAGE_URL)
table_a2_url, table_a2_name = find_resource_download_url(package_json, "Table A2")

raw_path = RAW_DIR / "apr_table_a2_raw.csv"
if not raw_path.exists():
    resp = requests.get(table_a2_url, timeout=300)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)

apr_raw = pd.read_csv(raw_path, low_memory=False)
print(apr_raw.shape)
apr_raw.head()


(921404, 69)


,JURIS_NAME,CNTY_NAME,YEAR,PRIOR_APN,APN,STREET_ADDRESS,PROJECT_NAME,JURS_TRACKING_ID,UNIT_CAT,TENURE,...,DEM_DES_UNITS_OWN_RENT,DENSITY_BONUS_TOTAL,DENSITY_BONUS_NUMBER_OTHER_INCENTIVES,DENSITY_BONUS_INCENTIVES,DENSITY_BONUS_RECEIVE_REDUCTION,NOTES,LATITUDE,LONGITUDE,STD_ADDRESS,SCORE
0,STANISLAUS COUNTY,Stanislaus,2020,NaN,038-048-002,627 HERNDON RD,NaN,BLD2020-0174,ADU,Renter,...,0,0.0,0,NaN,NaN,ACCESSORY DWELLING UNIT ( 1037 SF ) W/ ATTACHE...,37.620783,-120.975385,"627 Herndon Rd, Modesto, California, 95351",88.15
1,STANISLAUS COUNTY,Stanislaus,2020,NaN,038-040-030,907 BEWLEY AVE,NaN,BLD2019-2300,ADU,Renter,...,0,0.0,0,NaN,NaN,CONVERT GARAGE TO ACCESSORY DWELLING UNIT (ADU),37.616337,-120.979465,"907 Bewley Ave, Modesto, California, 95351",88.15
2,STANISLAUS COUNTY,Stanislaus,2020,NaN,012-032-002,5818 BECKWITH RD,NaN,BLD2020-1845,ADU,Renter,...,0,0.0,0,NaN,NaN,"1,116 SQ. FT. ACCESSORY DWELLING UNIT",37.681677,-121.102854,"5818 Beckwith Rd, Modesto, California, 95358",88.57
3,STANISLAUS COUNTY,Stanislaus,2020,NaN,035-037-012,1425,CANAL,BLD2019-0568,SFD,Renter,...,0,0.0,0,NaN,NaN,SINGLE FAMILY DWELLING ( 1092 SQ FT ) ( UNIT 3 ),37.559009,-120.997660,NaN,82.61
4,STANISLAUS COUNTY,Stanislaus,2020,NaN,024-067-026,3307,KALMAR,BLD2019-2872,SFD,Owner,...,0,0.0,0,NaN,NaN,SINGLE FAMILY DWELLING 1850 SF W/ ATTACHED 2 C...,37.559009,-120.997660,NaN,82.61


In [212]:
print(apr_raw.columns.tolist())

['JURIS_NAME', 'CNTY_NAME', 'YEAR', 'PRIOR_APN', 'APN', 'STREET_ADDRESS', 'PROJECT_NAME', 'JURS_TRACKING_ID', 'UNIT_CAT', 'TENURE', 'ACUTELY_LOW_INCOME_DR', 'ACUTELY_LOW_INCOME_NDR', 'EXTREMELY_LOW_INCOME_DR', 'EXTREMELY_LOW_INCOME_NDR', 'VLOW_INCOME_DR', 'VLOW_INCOME_NDR', 'LOW_INCOME_DR', 'LOW_INCOME_NDR', 'MOD_INCOME_DR', 'MOD_INCOME_NDR', 'ABOVE_MOD_INCOME', 'ENT_APPROVE_DT1', 'NO_ENTITLEMENTS', 'BP_ACUTELY_LOW_INCOME_DR', 'BP_ACUTELY_LOW_INCOME_NDR', 'BP_EXTREMELY_LOW_INCOME_DR', 'BP_EXTREMELY_LOW_INCOME_NDR', 'BP_VLOW_INCOME_DR', 'BP_VLOW_INCOME_NDR', 'BP_LOW_INCOME_DR', 'BP_LOW_INCOME_NDR', 'BP_MOD_INCOME_DR', 'BP_MOD_INCOME_NDR', 'BP_ABOVE_MOD_INCOME', 'BP_ISSUE_DT1', 'NO_BUILDING_PERMITS', 'CO_ACUTELY_LOW_INCOME_DR', 'CO_ACUTELY_LOW_INCOME_NDR', 'CO_EXTREMELY_LOW_INCOME_DR', 'CO_EXTREMELY_LOW_INCOME_NDR', 'CO_VLOW_INCOME_DR', 'CO_VLOW_INCOME_NDR', 'CO_LOW_INCOME_DR', 'CO_LOW_INCOME_NDR', 'CO_MOD_INCOME_DR', 'CO_MOD_INCOME_NDR', 'CO_ABOVE_MOD_INCOME', 'CO_ISSUE_DT1', 'NO_OTHER_

## APR production measures

For entitlement, building-permit, and completion measures, the notebook uses
the corresponding HCD stage date (`ENT_APPROVE_DT1`, `BP_ISSUE_DT1`,
`CO_ISSUE_DT1`) to ensure the activity occurred in the row's reporting year.

This matters because a Table A2 project can appear across multiple reporting
years as it moves through different development stages.


In [213]:
JURISDICTION_COL = "JURIS_NAME"
YEAR_COL = "YEAR"

apr_raw["jur_clean"] = apr_raw[JURISDICTION_COL].map(normalize_jurisdiction)
apr_raw[YEAR_COL] = pd.to_numeric(apr_raw[YEAR_COL], errors="coerce")

sd_apr = apr_raw[apr_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_target_year = sd_apr[sd_apr[YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_target_year))
missing = sd_jur_keys - set(sd_apr_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


SD rows, all years: 52515
SD rows, 2025: 7461


In [214]:
TIER_SUFFIX_GROUPS = {
    "very_low": [
        "ACUTELY_LOW_INCOME_DR", "ACUTELY_LOW_INCOME_NDR",
        "EXTREMELY_LOW_INCOME_DR", "EXTREMELY_LOW_INCOME_NDR",
        "VLOW_INCOME_DR", "VLOW_INCOME_NDR",
    ],
    "low": ["LOW_INCOME_DR", "LOW_INCOME_NDR"],
    "moderate": ["MOD_INCOME_DR", "MOD_INCOME_NDR"],
    "above_moderate": ["ABOVE_MOD_INCOME"],
}

ALL_INCOME_SUFFIXES = [
    suffix
    for suffixes in TIER_SUFFIX_GROUPS.values()
    for suffix in suffixes
]

ENT_INCOME_COLS = [
    c for c in ALL_INCOME_SUFFIXES
    if c in sd_apr.columns
]

BP_INCOME_COLS = [
    f"BP_{suffix}" for suffix in ALL_INCOME_SUFFIXES
    if f"BP_{suffix}" in sd_apr.columns
]

CO_INCOME_COLS = [
    f"CO_{suffix}" for suffix in ALL_INCOME_SUFFIXES
    if f"CO_{suffix}" in sd_apr.columns
]

above_mod_ent = [
    c for c in ENT_INCOME_COLS
    if "ABOVE_MOD_INCOME" in c
]

above_mod_bp = [
    c for c in BP_INCOME_COLS
    if "ABOVE_MOD_INCOME" in c
]

above_mod_co = [
    c for c in CO_INCOME_COLS
    if "ABOVE_MOD_INCOME" in c
]


def _numeric_row_sum(frame, columns):
    """Numeric row sum that is stable even if HCD reads a field as object."""
    if not columns:
        return pd.Series(0.0, index=frame.index)

    numeric = frame[columns].apply(
        pd.to_numeric,
        errors="coerce",
    )

    return numeric.fillna(0).sum(axis=1)


def add_reporting_year_stage_units(df):
    """
    Add entitlement / permit / completion unit counts that are valid for
    the row's APR reporting year.

    HCD Table A2 can retain information from multiple development stages.
    A stage is counted in a given APR year only when that stage's date falls
    within the row's YEAR.
    """
    out = df.copy()

    report_year = pd.to_numeric(
        out[YEAR_COL],
        errors="coerce",
    )

    stage_config = {
        "ent": {
            "date_col": "ENT_APPROVE_DT1",
            "all_cols": ENT_INCOME_COLS,
            "above_cols": above_mod_ent,
            "column_prefix": "",
        },
        "bp": {
            "date_col": "BP_ISSUE_DT1",
            "all_cols": BP_INCOME_COLS,
            "above_cols": above_mod_bp,
            "column_prefix": "BP_",
        },
        "co": {
            "date_col": "CO_ISSUE_DT1",
            "all_cols": CO_INCOME_COLS,
            "above_cols": above_mod_co,
            "column_prefix": "CO_",
        },
    }

    for stage, config in stage_config.items():
        date_col = config["date_col"]

        stage_date = pd.to_datetime(
            out[date_col],
            errors="coerce",
        )

        activity_in_report_year = (
            stage_date.dt.year.eq(report_year)
        )

        out[f"{stage}_activity_in_report_year"] = (
            activity_in_report_year
        )

        raw_total = _numeric_row_sum(
            out,
            config["all_cols"],
        )

        out[f"{stage}_units_raw_row"] = raw_total

        out[f"{stage}_units_row"] = raw_total.where(
            activity_in_report_year,
            0,
        )

        raw_above = _numeric_row_sum(
            out,
            config["above_cols"],
        )

        raw_affordable = raw_total - raw_above

        out[f"{stage}_affordable_row"] = raw_affordable.where(
            activity_in_report_year,
            0,
        )

        for tier, suffixes in TIER_SUFFIX_GROUPS.items():
            tier_columns = []

            for suffix in suffixes:
                candidate = f"{config['column_prefix']}{suffix}"

                if candidate in out.columns:
                    tier_columns.append(candidate)

            tier_raw = _numeric_row_sum(
                out,
                tier_columns,
            )

            out[f"{stage}_{tier}_row"] = tier_raw.where(
                activity_in_report_year,
                0,
            )

    return out


# Apply once to the full APR Table A2 history so both the target-year
# and historical aggregations use exactly the same stage-date rules.
sd_apr = add_reporting_year_stage_units(sd_apr)

sd_apr_target_year = sd_apr[
    sd_apr[YEAR_COL].eq(TARGET_YEAR)
].copy()

tier_agg_kwargs = {}

for stage in ["ent", "bp", "co"]:
    for tier in TIER_SUFFIX_GROUPS:
        tier_agg_kwargs[f"{stage}_{tier}_total"] = (
            f"{stage}_{tier}_row",
            "sum",
        )

production_by_year = (
    sd_apr_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
        **tier_agg_kwargs,
    )
)

production_by_year["ent_affordable_share"] = (
    production_by_year["ent_affordable_total"]
    / production_by_year["ent_units_total"]
)

production_by_year["bp_affordable_share"] = (
    production_by_year["bp_affordable_total"]
    / production_by_year["bp_units_total"]
)

production_by_year["co_affordable_share"] = (
    production_by_year["co_affordable_total"]
    / production_by_year["co_units_total"]
)


# QA: identify HCD rows that contain stage units but whose corresponding
# stage date is missing or outside the reporting year. These units are not
# counted in annual stage activity.
stage_date_qa_rows = []

for stage, date_col in {
    "ent": "ENT_APPROVE_DT1",
    "bp": "BP_ISSUE_DT1",
    "co": "CO_ISSUE_DT1",
}.items():

    stage_date = pd.to_datetime(
        sd_apr[date_col],
        errors="coerce",
    )

    raw_units = sd_apr[f"{stage}_units_raw_row"]

    activity_mask = sd_apr[
        f"{stage}_activity_in_report_year"
    ]

    stage_date_qa_rows.append(
        {
            "development_stage": stage,
            "rows_with_units": int((raw_units > 0).sum()),
            "rows_counted_in_reporting_year": int(
                ((raw_units > 0) & activity_mask).sum()
            ),
            "rows_with_units_missing_stage_date": int(
                ((raw_units > 0) & stage_date.isna()).sum()
            ),
            "rows_with_units_stage_date_outside_reporting_year": int(
                (
                    (raw_units > 0)
                    & stage_date.notna()
                    & (~activity_mask)
                ).sum()
            ),
            "units_excluded_from_annual_total_due_to_stage_date": float(
                raw_units.where(
                    (raw_units > 0) & (~activity_mask),
                    0,
                ).sum()
            ),
        }
    )

stage_date_qa = pd.DataFrame(stage_date_qa_rows)

print(production_by_year.shape)
display(stage_date_qa)
production_by_year


(19, 23)


,development_stage,rows_with_units,rows_counted_in_reporting_year,rows_with_units_missing_stage_date,rows_with_units_stage_date_outside_reporting_year,units_excluded_from_annual_total_due_to_stage_date
0,ent,3925,3605,0,320,704.0
1,bp,33550,32814,0,736,1996.0
2,co,21485,21314,0,171,228.0


,jur_clean,ent_units_total,bp_units_total,co_units_total,ent_affordable_total,bp_affordable_total,co_affordable_total,project_rows,ent_very_low_total,ent_low_total,...,bp_low_total,bp_moderate_total,bp_above_moderate_total,co_very_low_total,co_low_total,co_moderate_total,co_above_moderate_total,ent_affordable_share,bp_affordable_share,co_affordable_share
0,carlsbad,202,343,718,13,36,141,428,11,0,...,19,1,307,45,84,12,577,0.064356,0.104956,0.196379
1,chula vista,1323,717,1428,0,224,201,663,0,0,...,2,222,493,0,1,200,1227,0.000000,0.312413,0.140756
2,coronado,27,24,36,0,0,0,54,0,0,...,0,0,24,0,0,0,36,0.000000,0.000000,0.000000
3,del mar,9,14,17,8,11,10,43,0,0,...,0,11,3,0,0,10,7,0.888889,0.785714,0.588235
4,el cajon,99,210,235,40,83,123,164,0,4,...,38,45,127,0,99,24,112,0.404040,0.395238,0.523404
5,encinitas,277,187,228,40,33,43,461,0,31,...,2,14,154,25,3,15,185,0.144404,0.176471,0.188596
6,escondido,352,514,604,233,246,66,396,118,109,...,118,11,268,32,30,4,538,0.661932,0.478599,0.109272
7,imperial beach,45,71,13,0,0,0,86,0,0,...,0,0,71,0,0,0,13,0.000000,0.000000,0.000000
8,la mesa,78,254,249,11,99,215,223,0,0,...,22,69,155,60,93,62,34,0.141026,0.389764,0.863454
9,lemon grove,1,32,61,0,6,8,77,0,0,...,6,0,26,0,8,0,53,0.000000,0.187500,0.131148


### HCD auto-total validation

The validation below compares the notebook's stage totals with HCD's
`NO_ENTITLEMENTS`, `NO_BUILDING_PERMITS`, and
`NO_OTHER_FORMS_OF_READINESS` fields **after applying the same reporting-year
stage-date mask**. This avoids treating a prior-year project stage as current
annual activity.


In [215]:
HCD_STAGE_TOTAL_FIELDS = {
    "ent": "NO_ENTITLEMENTS",
    "bp": "NO_BUILDING_PERMITS",
    "co": "NO_OTHER_FORMS_OF_READINESS",
}

stage_validation_rows = []

for stage, hcd_total_field in HCD_STAGE_TOTAL_FIELDS.items():

    hcd_total_numeric = pd.to_numeric(
        sd_apr_target_year[hcd_total_field],
        errors="coerce",
    ).fillna(0)

    activity_mask = sd_apr_target_year[
        f"{stage}_activity_in_report_year"
    ]

    hcd_current_year_row = hcd_total_numeric.where(
        activity_mask,
        0,
    )

    ours_by_jur = (
        sd_apr_target_year
        .groupby("jur_clean")[
            f"{stage}_units_row"
        ]
        .sum()
    )

    hcd_by_jur = (
        pd.DataFrame(
            {
                "jur_clean": sd_apr_target_year["jur_clean"],
                "hcd_current_year_row": hcd_current_year_row,
            }
        )
        .groupby("jur_clean")["hcd_current_year_row"]
        .sum()
    )

    validation = pd.concat(
        [
            ours_by_jur.rename("ours"),
            hcd_by_jur.rename("hcd"),
        ],
        axis=1,
    ).reset_index()

    validation["development_stage"] = stage
    validation["difference"] = (
        validation["ours"] - validation["hcd"]
    )

    stage_validation_rows.append(validation)

stage_auto_total_validation = pd.concat(
    stage_validation_rows,
    ignore_index=True,
)

assert stage_auto_total_validation["difference"].eq(0).all(), (
    "One or more APR stage totals do not match HCD's auto-total fields "
    "after applying the reporting-year stage-date filter."
)

print("APR stage auto-total validation passed.")
stage_auto_total_validation


APR stage auto-total validation passed.


,jur_clean,ours,hcd,development_stage,difference
0,carlsbad,202,202,ent,0
1,chula vista,1323,1323,ent,0
2,coronado,27,27,ent,0
3,del mar,9,9,ent,0
4,el cajon,99,99,ent,0
5,encinitas,277,277,ent,0
6,escondido,352,352,ent,0
7,imperial beach,45,45,ent,0
8,la mesa,78,78,ent,0
9,lemon grove,1,1,ent,0


### Historical range check (2018-2025)

APR data collection began in 2018. Confirms every year is actually
present for San Diego County, not just assumed.

In [216]:
APR_START_YEAR = 2018
APR_YEARS = list(range(APR_START_YEAR, TARGET_YEAR + 1))

tier_agg_kwargs_hist = {}

for stage in ["ent", "bp", "co"]:
    for tier in TIER_SUFFIX_GROUPS:
        tier_agg_kwargs_hist[f"{stage}_{tier}_total"] = (
            f"{stage}_{tier}_row",
            "sum",
        )

production_history = (
    sd_apr[
        sd_apr[YEAR_COL].isin(APR_YEARS)
    ]
    .groupby(["jur_clean", YEAR_COL], as_index=False)
    .agg(
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
        **tier_agg_kwargs_hist,
    )
    .rename(columns={YEAR_COL: "year"})
)

production_history["ent_affordable_share"] = (
    production_history["ent_affordable_total"]
    / production_history["ent_units_total"]
)

production_history["bp_affordable_share"] = (
    production_history["bp_affordable_total"]
    / production_history["bp_units_total"]
)

production_history["co_affordable_share"] = (
    production_history["co_affordable_total"]
    / production_history["co_units_total"]
)

years_present = sorted(
    production_history["year"]
    .dropna()
    .unique()
)

years_missing = sorted(
    set(APR_YEARS)
    - set(years_present)
)

print("Years present:", years_present)

if years_missing:
    print("Years missing entirely:", years_missing)

coverage_by_year = (
    production_history
    .groupby("year")["jur_clean"]
    .agg(
        jurisdictions="nunique",
        rows="count",
    )
    .reindex(APR_YEARS)
)

coverage_by_year["missing_jurisdictions"] = (
    coverage_by_year["jurisdictions"]
    .apply(
        lambda n: 19 - n
        if pd.notna(n)
        else 19
    )
)

print(production_history.shape)
coverage_by_year


Years present: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
(152, 24)


,jurisdictions,rows,missing_jurisdictions
year,,,
2018,19,19,0
2019,19,19,0
2020,19,19,0
2021,19,19,0
2022,19,19,0
2023,19,19,0
2024,19,19,0
2025,19,19,0


In [217]:
years_present = sorted(production_history["year"].dropna().unique())
years_missing = sorted(set(APR_YEARS) - set(years_present))
print("Years present:", years_present)
if years_missing:
    print("Years missing entirely from the pull:", years_missing)

coverage = (
    production_history
    .groupby("year")["jur_clean"]
    .agg(jurisdictions="nunique", rows="count")
    .reindex(APR_YEARS)
)
coverage["missing_jurisdictions"] = coverage["jurisdictions"].apply(
    lambda n: 19 - n if pd.notna(n) else 19
)
coverage


Years present: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,jurisdictions,rows,missing_jurisdictions
year,,,
2018,19,19,0
2019,19,19,0
2020,19,19,0
2021,19,19,0
2022,19,19,0
2023,19,19,0
2024,19,19,0
2025,19,19,0


In [218]:
# Reindex AFTER the coverage check above, so the check reflects real
# reporting gaps - this fill is only for downstream use (Power BI export).
production_history = fill_missing_jurisdiction_years(production_history, APR_YEARS)
print(production_history.shape)


(152, 25)


In [219]:
history_output_path = PROCESSED_DIR / f"apr_production_{APR_START_YEAR}_{TARGET_YEAR}_by_jurisdiction_year.csv"
production_history.to_csv(history_output_path, index=False)
print("Saved:", history_output_path)


Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/apr_production_2018_2025_by_jurisdiction_year.csv


## APR Table A (applications)

Applications live in a separate table from entitlements/permits/completions
(Table A vs. Table A2). Loaded the same way as Table A2 - CKAN action API,
resolved by exact resource name.

In [220]:
table_a_package_json = get_json(CKAN_PACKAGE_URL)

table_a_matches = [
    r for r in table_a_package_json["result"]["resources"]
    if r.get("name", "").strip().lower() == "apr table a"
    and r.get("format", "").upper() == "CSV"
]
if not table_a_matches:
    raise ValueError("No exact 'APR Table A' CSV resource found in the package.")
table_a_url = table_a_matches[0]["url"]
table_a_name = table_a_matches[0]["name"]
print("Using resource:", table_a_name)
print("Download URL:", table_a_url)

table_a_raw_path = RAW_DIR / "apr_table_a_raw.csv"
if not table_a_raw_path.exists():
    resp = requests.get(table_a_url, timeout=300)
    resp.raise_for_status()
    table_a_raw_path.write_bytes(resp.content)

apr_table_a_raw = pd.read_csv(table_a_raw_path, low_memory=False)
print(apr_table_a_raw.shape)
apr_table_a_raw.head()


Using resource: APR Table A
Download URL: https://data.ca.gov/dataset/81b0841f-2802-403e-b48e-2ef4b751f77c/resource/c78b769d-cc02-4050-91ef-79ded665b5a8/download/tablea.csv
(357875, 36)


,JURIS_NAME,CNTY_NAME,YEAR,PRIOR_APN,APN,STREET_ADDRESS,PROJECT_NAME,JURS_TRACKING_ID,UNIT_CAT,TENURE,...,HISTORIC_SITES,DENSITY_BONUS_RECEIVED,DENSITY_BONUS_APPROVED,APPLICATION_STATUS,PROJECT_TYPE,NOTES,LATITUDE,LONGITUDE,STD_ADDRESS,SCORE
0,PETALUMA,Sonoma,2021,NaN,006-092-017,719 PETALUMA BLVD N\nPET...,NaN,NaN,2 to 4,Owner,...,NaN,No,No,Pending,NaN,Believe this remains in CUP/SPAR Review –waiti...,36.219404,-119.365743,"719 N Petaluma St, Tulare, California, 93274",82.84
1,PETALUMA,Sonoma,2021,NaN,008-490-031,450 HAYES LN\nPETALUM...,NaN,NaN,SFD,Owner,...,NaN,No,No,Pending,NaN,"In for BP review (submitted, not yet issued)",38.221759,-122.648132,"450 Hayes Ln, Petaluma, California, 94952",98.50
2,LONG BEACH,Los Angeles,2021,NaN,7138012029,1125 E Carson St,2108-17,2108-17,ADU,Renter,...,NaN,No,NaN,Approved,NaN,NaN,33.832448,-118.178164,"1125 E Carson St, Long Beach, California, 90807",100.00
3,LONG BEACH,Los Angeles,2021,NaN,7275010004,1212 E. 3rd Street,2111-13,2111-13,ADU,Renter,...,NaN,No,NaN,Pending,NaN,NaN,33.770077,-118.176200,"1212 E 3rd St, Long Beach, California, 90802",100.00
4,LONG BEACH,Los Angeles,2021,NaN,7267018001,1600 E 10th St,2103-43,2103-43,ADU,Renter,...,NaN,No,NaN,Pending,NaN,NaN,33.778943,-118.171687,"1600 E 10th St, Long Beach, California, 90813",100.00


In [221]:
print(apr_table_a_raw.columns.tolist())

['JURIS_NAME', 'CNTY_NAME', 'YEAR', 'PRIOR_APN', 'APN', 'STREET_ADDRESS', 'PROJECT_NAME', 'JURS_TRACKING_ID', 'UNIT_CAT', 'TENURE', 'APP_SUBMIT_DT', 'ACUTELY_LOW_INCOME_DR', 'ACUTELY_LOW_INCOME_NDR', 'EXTREMELY_LOW_INCOME_DR', 'EXTREMELY_LOW_INCOME_NDR', 'VLOW_INCOME_DR', 'VLOW_INCOME_NDR', 'LOW_INCOME_DR', 'LOW_INCOME_NDR', 'MOD_INCOME_DR', 'MOD_INCOME_NDR', 'ABOVE_MOD_INCOME', 'TOT_PROPOSED_UNITS', 'TOT_APPROVED_UNITS', 'TOT_DISAPPROVED_UNITS', 'APP_SUBMITTED_SB35', 'HISTORIC_SITES', 'DENSITY_BONUS_RECEIVED', 'DENSITY_BONUS_APPROVED', 'APPLICATION_STATUS', 'PROJECT_TYPE', 'NOTES', 'LATITUDE', 'LONGITUDE', 'STD_ADDRESS', 'SCORE']


In [222]:
TABLE_A_JUR_COL = "JURIS_NAME"
TABLE_A_YEAR_COL = "YEAR"

apr_table_a_raw["jur_clean"] = apr_table_a_raw[TABLE_A_JUR_COL].map(normalize_jurisdiction)
apr_table_a_raw[TABLE_A_YEAR_COL] = pd.to_numeric(apr_table_a_raw[TABLE_A_YEAR_COL], errors="coerce")

sd_apr_table_a = apr_table_a_raw[apr_table_a_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_table_a_target_year = sd_apr_table_a[sd_apr_table_a[TABLE_A_YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr_table_a))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_table_a_target_year))
missing = sd_jur_keys - set(sd_apr_table_a_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


SD rows, all years: 23118
SD rows, 2025: 4874


In [223]:
APPLICATION_INCOME_COLS = [
    "ACUTELY_LOW_INCOME_DR", "ACUTELY_LOW_INCOME_NDR",
    "EXTREMELY_LOW_INCOME_DR", "EXTREMELY_LOW_INCOME_NDR",
    "VLOW_INCOME_DR", "VLOW_INCOME_NDR",
    "LOW_INCOME_DR", "LOW_INCOME_NDR",
    "MOD_INCOME_DR", "MOD_INCOME_NDR",
    "ABOVE_MOD_INCOME",
]
above_mod_application = [c for c in APPLICATION_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_table_a_target_year["TOT_PROPOSED_UNITS"] = pd.to_numeric(
    sd_apr_table_a_target_year["TOT_PROPOSED_UNITS"], errors="coerce"
)

sd_apr_table_a_target_year["application_classified_row"] = (
    sd_apr_table_a_target_year[APPLICATION_INCOME_COLS].sum(axis=1, numeric_only=True)
)

sd_apr_table_a_target_year["application_units_row"] = (
    sd_apr_table_a_target_year["TOT_PROPOSED_UNITS"].where(
        sd_apr_table_a_target_year["TOT_PROPOSED_UNITS"].notna(),
        sd_apr_table_a_target_year["application_classified_row"],
    )
)

sd_apr_table_a_target_year["application_unclassified_row"] = (
    sd_apr_table_a_target_year["application_units_row"]
    - sd_apr_table_a_target_year["application_classified_row"]
).clip(lower=0)

sd_apr_table_a_target_year["application_affordable_row"] = (
    sd_apr_table_a_target_year["application_classified_row"]
    - sd_apr_table_a_target_year[above_mod_application].sum(axis=1, numeric_only=True)
)

application_tier_agg_kwargs = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    app_cols = [s for s in suffixes if s in sd_apr_table_a_target_year.columns]
    sd_apr_table_a_target_year[f"application_{tier}_row"] = (
        sd_apr_table_a_target_year[app_cols].sum(axis=1, numeric_only=True)
    )
    application_tier_agg_kwargs[f"application_{tier}_total"] = (
        f"application_{tier}_row", "sum"
    )

applications_by_jurisdiction = (
    sd_apr_table_a_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        application_units_total=("application_units_row", "sum"),
        application_classified_total=("application_classified_row", "sum"),
        application_unclassified_total=("application_unclassified_row", "sum"),
        application_affordable_total=("application_affordable_row", "sum"),
        application_rows=("jur_clean", "size"),
        **application_tier_agg_kwargs,
    )
)

applications_by_jurisdiction["application_affordable_share"] = (
    applications_by_jurisdiction["application_affordable_total"]
    / applications_by_jurisdiction["application_units_total"]
)

print(applications_by_jurisdiction.shape)
applications_by_jurisdiction


(19, 11)


,jur_clean,application_units_total,application_classified_total,application_unclassified_total,application_affordable_total,application_rows,application_very_low_total,application_low_total,application_moderate_total,application_above_moderate_total,application_affordable_share
0,carlsbad,813,813,0,114,103,9,104,1,699,0.140221
1,chula vista,2047,2057,0,560,428,131,99,330,1497,0.273571
2,coronado,45,45,0,0,36,0,0,0,45,0.000000
3,del mar,18,18,0,15,18,0,0,15,3,0.833333
4,el cajon,241,241,0,200,148,0,66,134,41,0.829876
5,encinitas,315,316,0,19,137,2,8,9,297,0.060317
6,escondido,466,554,0,232,93,118,108,6,322,0.497854
7,imperial beach,172,197,0,55,88,28,0,27,142,0.319767
8,la mesa,453,453,0,129,148,5,19,105,324,0.284768
9,lemon grove,90,90,0,9,60,0,9,0,81,0.100000


In [224]:
proposed_check = (
    sd_apr_table_a_target_year
    .groupby("jur_clean", as_index=False)
    .agg(tot_proposed_units=("TOT_PROPOSED_UNITS", lambda s: s.sum(min_count=1)))
)

check = applications_by_jurisdiction.merge(
    proposed_check, on="jur_clean", how="left"
)

check["diff"] = (
    check["application_units_total"] - check["tot_proposed_units"]
)

assert (
    check.loc[check["tot_proposed_units"].notna(), "diff"].fillna(0).eq(0).all()
), "Application totals do not reconcile to HCD TOT_PROPOSED_UNITS."

check[
    [
        "jur_clean",
        "application_units_total",
        "application_classified_total",
        "application_unclassified_total",
        "tot_proposed_units",
        "diff",
    ]
]


,jur_clean,application_units_total,application_classified_total,application_unclassified_total,tot_proposed_units,diff
0,carlsbad,813,813,0,813,0
1,chula vista,2047,2057,0,2047,0
2,coronado,45,45,0,45,0
3,del mar,18,18,0,18,0
4,el cajon,241,241,0,241,0
5,encinitas,315,316,0,315,0
6,escondido,466,554,0,466,0
7,imperial beach,172,197,0,172,0
8,la mesa,453,453,0,453,0
9,lemon grove,90,90,0,90,0


### Historical applications (2018-2025)

Same treatment as APR Table A2 - `sd_apr_table_a` already has all years
for San Diego County; aggregate the full range rather than just
`TARGET_YEAR`.

In [225]:
sd_apr_table_a["TOT_PROPOSED_UNITS"] = pd.to_numeric(
    sd_apr_table_a["TOT_PROPOSED_UNITS"], errors="coerce"
)

sd_apr_table_a["application_classified_row"] = (
    sd_apr_table_a[APPLICATION_INCOME_COLS].sum(axis=1, numeric_only=True)
)

sd_apr_table_a["application_units_row"] = (
    sd_apr_table_a["TOT_PROPOSED_UNITS"].where(
        sd_apr_table_a["TOT_PROPOSED_UNITS"].notna(),
        sd_apr_table_a["application_classified_row"],
    )
)

sd_apr_table_a["application_unclassified_row"] = (
    sd_apr_table_a["application_units_row"]
    - sd_apr_table_a["application_classified_row"]
).clip(lower=0)

sd_apr_table_a["application_affordable_row"] = (
    sd_apr_table_a["application_classified_row"]
    - sd_apr_table_a[above_mod_application].sum(axis=1, numeric_only=True)
)

application_tier_agg_kwargs_hist = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    app_cols = [s for s in suffixes if s in sd_apr_table_a.columns]
    sd_apr_table_a[f"application_{tier}_row"] = (
        sd_apr_table_a[app_cols].sum(axis=1, numeric_only=True)
    )
    application_tier_agg_kwargs_hist[f"application_{tier}_total"] = (
        f"application_{tier}_row", "sum"
    )

applications_history = (
    sd_apr_table_a[sd_apr_table_a[TABLE_A_YEAR_COL].isin(APR_YEARS)]
    .groupby(["jur_clean", TABLE_A_YEAR_COL], as_index=False)
    .agg(
        application_units_total=("application_units_row", "sum"),
        application_classified_total=("application_classified_row", "sum"),
        application_unclassified_total=("application_unclassified_row", "sum"),
        application_affordable_total=("application_affordable_row", "sum"),
        application_rows=("jur_clean", "size"),
        **application_tier_agg_kwargs_hist,
    )
    .rename(columns={TABLE_A_YEAR_COL: "year"})
)

applications_history["application_affordable_share"] = (
    applications_history["application_affordable_total"]
    / applications_history["application_units_total"]
)

years_present_app = sorted(applications_history["year"].dropna().unique())
years_missing_app = sorted(set(APR_YEARS) - set(years_present_app))
print("Years present (applications):", years_present_app)
if years_missing_app:
    print("Years missing entirely:", years_missing_app)

coverage_app = (
    applications_history
    .groupby("year")["jur_clean"]
    .agg(jurisdictions="nunique", rows="count")
    .reindex(APR_YEARS)
)
coverage_app["missing_jurisdictions"] = coverage_app["jurisdictions"].apply(
    lambda n: 19 - n if pd.notna(n) else 19
)

print(applications_history.shape)
coverage_app


Years present (applications): [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
(149, 12)


,jurisdictions,rows,missing_jurisdictions
year,,,
2018,18,18,1
2019,18,18,1
2020,18,18,1
2021,19,19,0
2022,19,19,0
2023,19,19,0
2024,19,19,0
2025,19,19,0


In [226]:
for yr in [2018, 2019, 2020]:
    present = set(applications_history[applications_history["year"] == yr]["jur_clean"])
    missing = sd_jur_keys - present
    if missing:
        print(f"{yr}: missing {sorted(missing)}")


2018: missing ['lemon grove']
2019: missing ['imperial beach']
2020: missing ['san marcos']


In [227]:
applications_history = fill_missing_jurisdiction_years(applications_history, APR_YEARS)
print(applications_history.shape)


(152, 13)


## RHNA 6th Cycle targets

In [228]:
RHNA_PACKAGE_URL = "https://data.ca.gov/api/3/action/package_show?id=rhna-progress-report"
rhna_package_json = get_json(RHNA_PACKAGE_URL)
rhna6_url, rhna6_name = find_resource_download_url(rhna_package_json, "6th Cycle RHNA Progress Report")

rhna_raw_path = RAW_DIR / "rhna6_progress_raw.csv"
if not rhna_raw_path.exists():
    resp = requests.get(rhna6_url, timeout=120)
    resp.raise_for_status()
    rhna_raw_path.write_bytes(resp.content)

rhna_raw = pd.read_csv(rhna_raw_path, low_memory=False)
print(rhna_raw.shape)
rhna_raw.head()


(539, 15)


,Jurisdiction,Planning Period,6th Cycle Started,VLI UNITS,RHNA VLI,VLI %,LI UNITS,RHNA LI,LI %,MOD UNITS,RHNA MOD,MOD %,ABOVE MOD UNITS,RHNA ABOVE MOD,ABOVE MOD %
0,ALAMEDA,01/31/2023 - 01/31/2031,True,155,1421,0.11,47,818,0.06,55,868,0.06,192,2246,0.09
1,AGOURA HILLS,10/15/2021 - 10/15/2029,True,44,127,0.35,10,72,0.14,6,55,0.11,277,64,4.33
2,AMERICAN CANYON,01/31/2023 - 01/31/2031,True,11,169,0.07,5,109,0.05,2,95,0.02,487,249,1.96
3,ALPINE COUNTY,08/31/2019 - 06/30/2024,True,0,1,0.00,0,1,0.00,2,0,0.00,33,0,0.00
4,ALHAMBRA,10/15/2021 - 10/15/2029,True,21,1774,0.01,26,1036,0.03,1,1079,0.00,636,2936,0.22


In [229]:
print(rhna_raw.columns.tolist())

['Jurisdiction', 'Planning Period', '6th Cycle Started', 'VLI UNITS', 'RHNA VLI', 'VLI %', 'LI UNITS', 'RHNA LI', 'LI %', 'MOD UNITS', 'RHNA MOD', 'MOD %', 'ABOVE MOD UNITS', 'RHNA ABOVE MOD', 'ABOVE MOD %']


**RHNA progress basis:** `rhna_reported_<tier>` / `rhna_pct_achieved_<tier>`
/ `rhna_remaining_<tier>` are based on **building permits issued**, per
HCD's own RHNA-credit methodology - not completions, not entitlements.
Distinct from `co_units_total` (completions); don't conflate the two.

In [230]:
RHNA_JUR_COL = "Jurisdiction"

rhna_raw["jur_clean"] = rhna_raw[RHNA_JUR_COL].map(normalize_jurisdiction)
sd_rhna6 = rhna_raw[rhna_raw["jur_clean"].isin(sd_jur_keys)].copy()

print("SD rows:", len(sd_rhna6))
missing = sd_jur_keys - set(sd_rhna6["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_rhna6.head()


SD rows: 19


,Jurisdiction,Planning Period,6th Cycle Started,VLI UNITS,RHNA VLI,VLI %,LI UNITS,RHNA LI,LI %,MOD UNITS,RHNA MOD,MOD %,ABOVE MOD UNITS,RHNA ABOVE MOD,ABOVE MOD %,jur_clean
47,CARLSBAD,04/30/2021 - 04/30/2029,True,65,1311,0.05,198,784,0.25,293,749,0.39,991,1029,0.96,carlsbad
76,DEL MAR,04/30/2021 - 04/30/2029,True,0,37,0.00,0,64,0.00,66,31,2.13,45,31,1.45,del mar
104,EL CAJON,04/30/2021 - 04/30/2029,True,0,481,0.00,281,414,0.68,154,518,0.30,350,1867,0.19,el cajon
121,CORONADO,04/30/2021 - 04/30/2029,True,0,312,0.00,0,169,0.00,0,159,0.00,204,272,0.75,coronado
144,CHULA VISTA,04/30/2021 - 04/30/2029,True,130,2750,0.05,377,1777,0.21,943,1911,0.49,4716,4667,1.01,chula vista


## CA DOF population & housing estimates (E-5)

### DOF workbook (2026 release, using the revised 1/1/2025 estimates)

The official `E-5_2026_InternetVersion.xlsx` workbook contains the required `E5CityCounty2025` sheet. This notebook uses that 2025 sheet, not the 2026 sheet.

The cell below looks for the workbook in either:
- `van's work/data/raw/dof/E-5_2026_InternetVersion.xlsx`, or
- your Mac `~/Downloads/E-5_2026_InternetVersion.xlsx` folder.

If it finds the file in Downloads, it automatically copies it into the project data folder. No web scraping or automated DOF-site download is used.


In [231]:
from pathlib import Path
import shutil

# ============================================================
# California DOF E-5
# Official 2026 release, extracting the revised 1/1/2025 sheet
# ============================================================

DOF_SOURCE_RELEASE = 2026
DOF_ESTIMATE_YEAR = TARGET_YEAR

DOF_FILENAME = "E-5_2026_InternetVersion.xlsx"
DOF_RAW_PATH = RAW_DIR.parent / "dof" / DOF_FILENAME
DOF_DOWNLOADS_PATH = Path.home() / "Downloads" / DOF_FILENAME
DOF_SHEET_NAME = f"E5CityCounty{DOF_ESTIMATE_YEAR}"

DOF_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)


def verify_dof_workbook(path):
    """Verify that this is the expected DOF workbook and contains the 2025 sheet."""
    try:
        workbook = pd.ExcelFile(path)
        return DOF_SHEET_NAME in workbook.sheet_names
    except Exception:
        return False


# ------------------------------------------------------------
# Find the workbook locally.
# Prefer the project copy; otherwise copy it from ~/Downloads.
# ------------------------------------------------------------

if DOF_RAW_PATH.exists() and verify_dof_workbook(DOF_RAW_PATH):
    print("Using verified DOF workbook:")
    print(DOF_RAW_PATH)

elif DOF_DOWNLOADS_PATH.exists() and verify_dof_workbook(DOF_DOWNLOADS_PATH):
    shutil.copy2(DOF_DOWNLOADS_PATH, DOF_RAW_PATH)

    print("Found verified DOF workbook in Downloads:")
    print(DOF_DOWNLOADS_PATH)
    print()
    print("Copied to project data folder:")
    print(DOF_RAW_PATH)

else:
    raise FileNotFoundError(
        "Could not find the official DOF workbook "
        f"'{DOF_FILENAME}'. Put it in either:\n"
        f"  1. {DOF_RAW_PATH}\n"
        f"  2. {DOF_DOWNLOADS_PATH}\n"
        f"The workbook must contain the sheet '{DOF_SHEET_NAME}'."
    )


# ============================================================
# Read the REVISED 1/1/2025 estimate from the 2026 workbook
# ============================================================

dof_raw = pd.read_excel(
    DOF_RAW_PATH,
    sheet_name=DOF_SHEET_NAME,
    header=3,
)

dof_raw = dof_raw.rename(columns={"County/City/State": "name"})
dof_raw["name"] = dof_raw["name"].astype(str).str.strip()

# The sheet has two columns both labeled "Total":
# - population total
# - housing-unit total
# pandas disambiguates the second as "Total.1".
dof_raw = dof_raw.rename(
    columns={
        "Total": "population_total",
        "Total.1": "housing_units_total",
    }
)

required_dof_columns = {
    "name",
    "population_total",
    "Household",
    "Group Quarters",
    "housing_units_total",
    "Single Detached",
    "Single Attached",
    "Two to Four",
    "Five Plus",
    "Mobile Homes",
    "Occupied",
    "Vacancy Rate",
    "Persons per Household",
}

missing_dof_columns = required_dof_columns - set(dof_raw.columns)

assert not missing_dof_columns, (
    "DOF workbook structure changed. Missing columns: "
    f"{sorted(missing_dof_columns)}"
)

# County heading rows have a county name but no population total.
dof_raw["is_county_header"] = (
    dof_raw["population_total"].isna()
    & dof_raw["name"].str.contains("County", na=False)
)

dof_raw["county"] = (
    dof_raw["name"]
    .where(dof_raw["is_county_header"])
    .ffill()
)

# Keep the 18 incorporated cities plus Unincorporated San Diego County.
sd_dof = dof_raw[
    (dof_raw["county"] == "San Diego County")
    & (~dof_raw["is_county_header"])
    & (dof_raw["population_total"].notna())
    & (~dof_raw["name"].isin(["Incorporated", "County Total"]))
].copy()

sd_dof["jur_clean"] = sd_dof["name"].map(
    lambda n: (
        normalize_jurisdiction(COUNTY_JURISDICTION_NAME)
        if n.strip() == "Unincorporated"
        else normalize_jurisdiction(n)
    )
)

sd_dof["year"] = DOF_ESTIMATE_YEAR

sd_dof["vacant_units"] = (
    sd_dof["housing_units_total"] - sd_dof["Occupied"]
)

sd_dof["single_family_units"] = (
    sd_dof["Single Detached"] + sd_dof["Single Attached"]
)

sd_dof["multifamily_units"] = (
    sd_dof["Two to Four"] + sd_dof["Five Plus"]
)

sd_dof["mobile_home_units"] = sd_dof["Mobile Homes"]

structure_sum = (
    sd_dof["single_family_units"]
    + sd_dof["multifamily_units"]
    + sd_dof["mobile_home_units"]
)

assert (structure_sum == sd_dof["housing_units_total"]).all(), (
    "DOF structure-type columns do not sum to housing_units_total."
)

print()
print(
    f"Loaded DOF {DOF_ESTIMATE_YEAR} estimates "
    f"from the {DOF_SOURCE_RELEASE} E-5 release."
)
print("Sheet:", DOF_SHEET_NAME)
print("San Diego rows:", len(sd_dof))

missing = sd_jur_keys - set(sd_dof["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))

sd_dof[
    [
        "name",
        "population_total",
        "Household",
        "Group Quarters",
        "housing_units_total",
        "Occupied",
        "vacant_units",
        "single_family_units",
        "multifamily_units",
        "mobile_home_units",
        "Vacancy Rate",
        "Persons per Household",
    ]
]


Using verified DOF workbook:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/raw/dof/E-5_2026_InternetVersion.xlsx

Loaded DOF 2025 estimates from the 2026 E-5 release.
Sheet: E5CityCounty2025
San Diego rows: 19


,name,population_total,Household,Group Quarters,housing_units_total,Occupied,vacant_units,single_family_units,multifamily_units,mobile_home_units,Vacancy Rate,Persons per Household
615,Carlsbad,116022.0,115034.0,988.0,48888.0,45710.0,3178.0,33861.0,13817.0,1210.0,0.065006,2.516605
616,Chula Vista,281850.0,280211.0,1639.0,91485.0,88373.0,3112.0,56792.0,30800.0,3893.0,0.034017,3.170776
617,Coronado,22687.0,17283.0,5404.0,9646.0,7438.0,2208.0,5463.0,4180.0,3.0,0.228903,2.323608
618,Del Mar,3937.0,3937.0,0.0,2641.0,1952.0,689.0,1903.0,738.0,0.0,0.260886,2.016906
619,El Cajon,105449.0,102949.0,2500.0,37011.0,35686.0,1325.0,17276.0,17852.0,1883.0,0.0358,2.884857
620,Encinitas,62392.0,61835.0,557.0,27118.0,25046.0,2072.0,20958.0,5522.0,638.0,0.076407,2.468857
621,Escondido,151932.0,149374.0,2558.0,50883.0,49041.0,1842.0,29260.0,17918.0,3705.0,0.036201,3.045900
622,Imperial Beach,26362.0,25995.0,367.0,10269.0,9499.0,770.0,4934.0,5033.0,302.0,0.074983,2.736604
623,La Mesa,61863.0,61169.0,694.0,26851.0,25814.0,1037.0,14068.0,12615.0,168.0,0.038621,2.369606
624,Lemon Grove,28445.0,28080.0,365.0,9812.0,9462.0,350.0,7310.0,2424.0,78.0,0.035671,2.967660


In [232]:
assert DOF_SHEET_NAME == "E5CityCounty2025", (
    f"Expected to use the 2025 estimate sheet, got {DOF_SHEET_NAME}."
)

assert len(sd_dof) == 19, (
    f"Expected 18 cities + Unincorporated San Diego County = 19 rows, "
    f"got {len(sd_dof)}"
)

assert not (
    sd_jur_keys - set(sd_dof["jur_clean"].unique())
), "One or more San Diego jurisdictions are missing from the DOF data."

# Known values from the uploaded official 2026 workbook's E5CityCounty2025 sheet.
# These checks make sure the correct sheet/release is being read.
_dof_spot_checks = {
    "san diego": {
        "population_total": 1409429,
        "housing_units_total": 581050,
        "Occupied": 541963,
    },
    "unincorporated san diego county": {
        "population_total": 511798,
        "housing_units_total": 183336,
        "Occupied": 172330,
    },
}

for jurisdiction, expected in _dof_spot_checks.items():
    row = sd_dof.loc[sd_dof["jur_clean"].eq(jurisdiction)].iloc[0]

    for column, expected_value in expected.items():
        actual_value = row[column]
        assert actual_value == expected_value, (
            f"DOF spot-check failed for {jurisdiction}, {column}: "
            f"expected {expected_value}, got {actual_value}"
        )

print(
    "DOF validation passed: 19 jurisdictions and 2025 spot-check values "
    "match the official 2026-release workbook."
)


DOF validation passed: 19 jurisdictions and 2025 spot-check values match the official 2026-release workbook.


## Census ACS (2020-2024 5-year estimates)

In [233]:
from getpass import getpass

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

CENSUS_API_KEY = getpass("Census API key: ").strip()
if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

ACS_VARS = {
    "B01003_001E": "population_total",
    "B25001_001E": "housing_units_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
}

acs_url = f"https://api.census.gov/data/{ACS_DATA_YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_raw = get_json(acs_url, params=params)
acs_df = pd.DataFrame(acs_raw[1:], columns=acs_raw[0]).rename(columns=ACS_VARS)
acs_df["jur_clean"] = acs_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs = acs_df[acs_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"SD rows ({ACS_VINTAGE_LABEL}):", len(sd_acs))
sd_acs.head()


SD rows (2020-2024): 18


,NAME,population_total,housing_units_total,owner_occupied,renter_occupied,state,place,jur_clean
218,"Carlsbad city, California",114373,47314,27688,16350,06,11194,carlsbad
271,"Chula Vista city, California",276375,90273,51281,34429,06,13392,chula vista
315,"Coronado city, California",19015,9896,3991,3312,06,16378,coronado
364,"Del Mar city, California",3903,2550,987,868,06,18506,del mar
439,"El Cajon city, California",104449,35185,14066,19824,06,21712,el cajon


## Summary

In [234]:
# City of SD permits is intentionally not included here - it's an
# optional appendix section that runs later, not part of the core pipeline.
loaded = {
    "APR (permits/completions)": sd_apr_target_year if "sd_apr_target_year" in dir() else pd.DataFrame(),
    "RHNA6 (targets)": sd_rhna6 if "sd_rhna6" in dir() else pd.DataFrame(),
    "DOF (population/housing)": sd_dof if "sd_dof" in dir() else pd.DataFrame(),
    "ACS": sd_acs if "sd_acs" in dir() else pd.DataFrame(),
}
for name, df in loaded.items():
    if df.empty:
        print(f"{name:30s} not loaded")
    else:
        n_missing = (df.isna().sum() > 0).sum()
        print(f"{name:30s} {df.shape[0]} rows, {df.shape[1]} cols, {n_missing} cols with missing values")


APR (permits/completions)      7461 rows, 94 cols, 12 cols with missing values
RHNA6 (targets)                19 rows, 16 cols, 0 cols with missing values
DOF (population/housing)       19 rows, 21 cols, 0 cols with missing values
ACS                            18 rows, 8 cols, 0 cols with missing values


## Housing production categories & housing stock fields

In [235]:
production_categories = pd.DataFrame([
    {"category": "Application submitted", "income_tier": "Acutely Low through Above Moderate (6 tiers)", "source": "APR Table A", "field": "ACUTELY_LOW_INCOME_DR/_NDR ... ABOVE_MOD_INCOME (same 11-column pattern as Table A2's entitlement section)"},
    {"category": "Entitlement", "income_tier": "Very Low / Low / Moderate / Above Moderate", "source": "APR Table A2", "field": "Unprefixed income-unit fields: ACUTELY_LOW_INCOME_DR/_NDR, EXTREMELY_LOW_INCOME_DR/_NDR, VLOW_INCOME_DR/_NDR, LOW_INCOME_DR/_NDR, MOD_INCOME_DR/_NDR, ABOVE_MOD_INCOME"},
    {"category": "Building permit", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "BP_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "BP_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Very Low", "source": "APR Table A2", "field": "BP_VLOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Low", "source": "APR Table A2", "field": "BP_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Moderate", "source": "APR Table A2", "field": "BP_MOD_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "BP_ABOVE_MOD_INCOME"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "CO_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "CO_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Very Low", "source": "APR Table A2", "field": "CO_VLOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Low", "source": "APR Table A2", "field": "CO_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Moderate", "source": "APR Table A2", "field": "CO_MOD_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "CO_ABOVE_MOD_INCOME"},
    {"category": "RHNA target", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD"},
    {"category": "RHNA progress", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS"},
    {"category": "City permit approval", "income_tier": "Extremely Low / Very Low / Low / Moderate / Above Moderate", "source": "City of SD permits", "field": "APPROVAL_DU_EXTREMELY_LOW ... APPROVAL_DU_ABOVE_MODERATE"},
    {"category": "ADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_ADU_TOTAL (plus per-tier APPROVAL_ADU_* columns)"},
    {"category": "JADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_JADU_TOTAL (plus per-tier APPROVAL_JADU_* columns)"},
    {"category": "Preservation (existing affordable units retained)", "income_tier": "n/a", "source": "APR Table F (not yet loaded in this notebook)", "field": "see sd_apr_f_preservation_city_year.csv in dashboard prototype"},
])
production_categories.to_csv(DOCS_DIR / "housing_production_categories.csv", index=False)
production_categories


,category,income_tier,source,field
0,Application submitted,Acutely Low through Above Moderate (6 tiers),APR Table A,ACUTELY_LOW_INCOME_DR/_NDR ... ABOVE_MOD_INCOM...
1,Entitlement,Very Low / Low / Moderate / Above Moderate,APR Table A2,Unprefixed income-unit fields: ACUTELY_LOW_INC...
2,Building permit,Acutely Low,APR Table A2,BP_ACUTELY_LOW_INCOME_DR / _NDR
3,Building permit,Extremely Low,APR Table A2,BP_EXTREMELY_LOW_INCOME_DR / _NDR
4,Building permit,Very Low,APR Table A2,BP_VLOW_INCOME_DR / _NDR
5,Building permit,Low,APR Table A2,BP_LOW_INCOME_DR / _NDR
6,Building permit,Moderate,APR Table A2,BP_MOD_INCOME_DR / _NDR
7,Building permit,Above Moderate,APR Table A2,BP_ABOVE_MOD_INCOME
8,Completion (certificate of occupancy),Acutely Low,APR Table A2,CO_ACUTELY_LOW_INCOME_DR / _NDR
9,Completion (certificate of occupancy),Extremely Low,APR Table A2,CO_EXTREMELY_LOW_INCOME_DR / _NDR


In [236]:
stock_fields = pd.DataFrame([
    {"field": "Total population", "source": "DOF E-5", "column": "population_total (raw: \"Total\", disambiguated from housing_units_total)"},
    {"field": "Household population", "source": "DOF E-5", "column": "Household"},
    {"field": "Group quarters population", "source": "DOF E-5", "column": "Group Quarters"},
    {"field": "Total housing units", "source": "DOF E-5", "column": "housing_units_total (raw: \"Total.1\")"},
    {"field": "Occupied units", "source": "DOF E-5", "column": "occupied_units (raw: \"Occupied\")"},
    {"field": "Vacant units", "source": "DOF E-5", "column": "vacant_units (computed: housing_units_total - Occupied)"},
    {"field": "Single-family units", "source": "DOF E-5", "column": "single_family_units (computed: Single Detached + Single Attached)"},
    {"field": "Multifamily units", "source": "DOF E-5", "column": "multifamily_units (computed: Two to Four + Five Plus)"},
    {"field": "Mobile home units", "source": "DOF E-5", "column": "mobile_home_units (raw: Mobile Homes)"},
    {"field": "Single detached units (detail)", "source": "DOF E-5", "column": "Single Detached"},
    {"field": "Single attached units (detail)", "source": "DOF E-5", "column": "Single Attached"},
    {"field": "2-4 unit buildings (detail)", "source": "DOF E-5", "column": "Two to Four"},
    {"field": "5+ unit buildings (detail)", "source": "DOF E-5", "column": "Five Plus"},
    {"field": "Vacancy rate", "source": "DOF E-5", "column": "Vacancy Rate"},
    {"field": "Persons per household", "source": "DOF E-5", "column": "Persons per Household"},
    {"field": "Total population", "source": "ACS 5-year", "column": "B01003_001E (population_total)"},
    {"field": "Total housing units", "source": "ACS 5-year", "column": "B25001_001E (housing_units_total)"},
    {"field": "Owner-occupied units", "source": "ACS 5-year", "column": "B25003_002E (owner_occupied)"},
    {"field": "Renter-occupied units", "source": "ACS 5-year", "column": "B25003_003E (renter_occupied)"},
])
stock_fields.to_csv(DOCS_DIR / "housing_stock_fields.csv", index=False)
stock_fields


,field,source,column
0,Total population,DOF E-5,"population_total (raw: ""Total"", disambiguated ..."
1,Household population,DOF E-5,Household
2,Group quarters population,DOF E-5,Group Quarters
3,Total housing units,DOF E-5,"housing_units_total (raw: ""Total.1"")"
4,Occupied units,DOF E-5,"occupied_units (raw: ""Occupied"")"
5,Vacant units,DOF E-5,vacant_units (computed: housing_units_total - ...
6,Single-family units,DOF E-5,single_family_units (computed: Single Detached...
7,Multifamily units,DOF E-5,multifamily_units (computed: Two to Four + Fiv...
8,Mobile home units,DOF E-5,mobile_home_units (raw: Mobile Homes)
9,Single detached units (detail),DOF E-5,Single Detached


**Notes on coverage:**

- DOF and ACS both report total population/housing units, but DOF breaks
  housing units out by structure type (single/multi/mobile) while ACS
  breaks out tenure (owner/renter) instead - they're complementary, not
  duplicates, unlike the APR/City-permits overlap found below.
- The APR field names use `VLOW` for "Very Low" while RHNA6 uses `VLI` -
  same tier, different abbreviation between the two datasets.
- DR/NDR suffixes on APR income columns = Deed Restricted / Non-Deed
  Restricted (whether the affordability requirement is legally recorded
  on the property).
- Preservation (Table F) is not yet loaded into this notebook - it exists
  as a processed file in the dashboard prototype repo but isn't part of
  this workstream's live pulls yet.


## Jurisdiction-year RHNA and housing production dataset

In [237]:
# RHNA6 has no year column (cumulative cycle-to-date, not annual), so its
# target/progress numbers get joined onto each jurisdiction as static
# context rather than matched by year.
#
# Kept broken out by income category (not just totals).
RHNA_TIERS = {
    "very_low": ("RHNA VLI", "VLI UNITS"),
    "low": ("RHNA LI", "LI UNITS"),
    "moderate": ("RHNA MOD", "MOD UNITS"),
    "above_moderate": ("RHNA ABOVE MOD", "ABOVE MOD UNITS"),
}

rhna_summary = sd_rhna6.copy()
tier_cols = ["jur_clean"]

for tier, (target_col, reported_col) in RHNA_TIERS.items():
    rhna_summary[f"rhna_target_{tier}"] = rhna_summary[target_col]
    rhna_summary[f"rhna_reported_{tier}"] = rhna_summary[reported_col]
    # Clipped at 0 - overachieving one tier doesn't offset a shortfall in
    # another; each income tier is a separate obligation, not a shared pool.
    rhna_summary[f"rhna_remaining_{tier}"] = (
        rhna_summary[f"rhna_target_{tier}"] - rhna_summary[f"rhna_reported_{tier}"]
    ).clip(lower=0)
    rhna_summary[f"rhna_pct_achieved_{tier}"] = rhna_summary[f"rhna_reported_{tier}"] / rhna_summary[f"rhna_target_{tier}"]
    tier_cols += [
        f"rhna_target_{tier}", f"rhna_reported_{tier}",
        f"rhna_remaining_{tier}", f"rhna_pct_achieved_{tier}",
    ]

rhna_summary["rhna_target_total"] = rhna_summary[["RHNA VLI", "RHNA LI", "RHNA MOD", "RHNA ABOVE MOD"]].sum(axis=1)
rhna_summary["rhna_units_reported_total"] = rhna_summary[["VLI UNITS", "LI UNITS", "MOD UNITS", "ABOVE MOD UNITS"]].sum(axis=1)
rhna_summary["rhna_remaining_total"] = rhna_summary[[f"rhna_remaining_{t}" for t in RHNA_TIERS]].sum(axis=1)
rhna_summary["rhna_pct_achieved"] = rhna_summary["rhna_units_reported_total"] / rhna_summary["rhna_target_total"]
tier_cols += ["rhna_target_total", "rhna_units_reported_total", "rhna_remaining_total", "rhna_pct_achieved"]

rhna_summary = rhna_summary[tier_cols]

dof_summary = sd_dof.rename(columns={"Household": "population_household"})[[
    "jur_clean", "population_total", "population_household",
    "housing_units_total", "Occupied", "vacant_units",
    "single_family_units", "multifamily_units", "mobile_home_units",
]].rename(columns={"Occupied": "occupied_units"})

jurisdiction_year_dataset = (
    production_by_year
    .merge(applications_by_jurisdiction, on="jur_clean", how="left")
    .merge(rhna_summary, on="jur_clean", how="left")
    .merge(dof_summary, on="jur_clean", how="left")
)

print(jurisdiction_year_dataset.shape)
jurisdiction_year_dataset


(19, 61)


,jur_clean,ent_units_total,bp_units_total,co_units_total,ent_affordable_total,bp_affordable_total,co_affordable_total,project_rows,ent_very_low_total,ent_low_total,...,rhna_remaining_total,rhna_pct_achieved,population_total,population_household,housing_units_total,occupied_units,vacant_units,single_family_units,multifamily_units,mobile_home_units
0,carlsbad,202,343,718,13,36,141,428,11,0,...,2326,0.399432,116022.0,115034.0,48888.0,45710.0,3178.0,33861.0,13817.0,1210.0
1,chula vista,1323,717,1428,0,224,201,663,0,0,...,4988,0.555245,281850.0,280211.0,91485.0,88373.0,3112.0,56792.0,30800.0,3893.0
2,coronado,27,24,36,0,0,0,54,0,0,...,708,0.223684,22687.0,17283.0,9646.0,7438.0,2208.0,5463.0,4180.0,3.0
3,del mar,9,14,17,8,11,10,43,0,0,...,101,0.680982,3937.0,3937.0,2641.0,1952.0,689.0,1903.0,738.0,0.0
4,el cajon,99,210,235,40,83,123,164,0,4,...,2495,0.239329,105449.0,102949.0,37011.0,35686.0,1325.0,17276.0,17852.0,1883.0
5,encinitas,277,187,228,40,33,43,461,0,31,...,832,0.932432,62392.0,61835.0,27118.0,25046.0,2072.0,20958.0,5522.0,638.0
6,escondido,352,514,604,233,246,66,396,118,109,...,7503,0.219007,151932.0,149374.0,50883.0,49041.0,1842.0,29260.0,17918.0,3705.0
7,imperial beach,45,71,13,0,0,0,86,0,0,...,1116,0.160271,26362.0,25995.0,10269.0,9499.0,770.0,4934.0,5033.0,302.0
8,la mesa,78,254,249,11,99,215,223,0,0,...,2741,0.278114,61863.0,61169.0,26851.0,25814.0,1037.0,14068.0,12615.0,168.0
9,lemon grove,1,32,61,0,6,8,77,0,0,...,1010,0.256806,28445.0,28080.0,9812.0,9462.0,350.0,7310.0,2424.0,78.0


In [238]:
missing = sd_jur_keys - set(jurisdiction_year_dataset["jur_clean"].unique())
if missing:
    print("Jurisdictions missing from the combined table:", sorted(missing))
else:
    print("All 18 cities + Unincorporated San Diego County present.")

null_counts = jurisdiction_year_dataset.isna().sum()
null_counts[null_counts > 0]


All 18 cities + Unincorporated San Diego County present.


Series([], dtype: int64)

## San Diego regional total

Separate object, not a 20th row in `jurisdiction_year_dataset` - keeps
countywide and jurisdiction-level results distinct. Sums all 18 cities
plus the unincorporated county.

In [239]:
assert len(jurisdiction_year_dataset) == 19, (
    f"Expected 18 cities + unincorporated county = 19 rows, got {len(jurisdiction_year_dataset)}"
)

RHNA_TIER_NAMES = ["very_low", "low", "moderate", "above_moderate"]
PRODUCTION_TIER_NAMES = ["very_low", "low", "moderate", "above_moderate"]

SUM_COLS = [
    "application_units_total",
    "application_classified_total",
    "application_unclassified_total",
    "ent_units_total",
    "bp_units_total",
    "co_units_total",
    "application_affordable_total",
    "ent_affordable_total",
    "bp_affordable_total",
    "co_affordable_total",
    "project_rows",
    "rhna_target_total",
    "rhna_units_reported_total",
    "rhna_remaining_total",
    "population_total",
    "population_household",
    "housing_units_total",
    "occupied_units",
    "vacant_units",
    "single_family_units",
    "multifamily_units",
    "mobile_home_units",
]

SUM_COLS += [f"application_{t}_total" for t in PRODUCTION_TIER_NAMES]
SUM_COLS += [f"ent_{t}_total" for t in PRODUCTION_TIER_NAMES]
SUM_COLS += [f"bp_{t}_total" for t in PRODUCTION_TIER_NAMES]
SUM_COLS += [f"co_{t}_total" for t in PRODUCTION_TIER_NAMES]
SUM_COLS += [f"rhna_target_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"rhna_reported_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"rhna_remaining_{t}" for t in RHNA_TIER_NAMES]

sd_region_total = jurisdiction_year_dataset[SUM_COLS].sum(
    min_count=len(jurisdiction_year_dataset)
).to_frame().T
sd_region_total.insert(
    0, "jurisdiction", "San Diego Region (18 cities + Unincorporated County)"
)
sd_region_total.insert(1, "year", TARGET_YEAR)
sd_region_total.insert(2, "jurisdictions_included", len(jurisdiction_year_dataset))

# Recalculate shares/percentages at the regional level rather than summing
# or averaging jurisdiction percentages.
sd_region_total["application_affordable_share"] = (
    sd_region_total["application_affordable_total"]
    / sd_region_total["application_units_total"]
)
sd_region_total["ent_affordable_share"] = (
    sd_region_total["ent_affordable_total"]
    / sd_region_total["ent_units_total"]
)
sd_region_total["bp_affordable_share"] = (
    sd_region_total["bp_affordable_total"]
    / sd_region_total["bp_units_total"]
)
sd_region_total["co_affordable_share"] = (
    sd_region_total["co_affordable_total"]
    / sd_region_total["co_units_total"]
)
sd_region_total["rhna_pct_achieved"] = (
    sd_region_total["rhna_units_reported_total"]
    / sd_region_total["rhna_target_total"]
)

for tier in RHNA_TIER_NAMES:
    sd_region_total[f"rhna_pct_achieved_{tier}"] = (
        sd_region_total[f"rhna_reported_{tier}"]
        / sd_region_total[f"rhna_target_{tier}"]
    )

sd_region_total


,jurisdiction,year,jurisdictions_included,application_units_total,application_classified_total,application_unclassified_total,ent_units_total,bp_units_total,co_units_total,application_affordable_total,...,rhna_remaining_above_moderate,application_affordable_share,ent_affordable_share,bp_affordable_share,co_affordable_share,rhna_pct_achieved,rhna_pct_achieved_very_low,rhna_pct_achieved_low,rhna_pct_achieved_moderate,rhna_pct_achieved_above_moderate
0,San Diego Region (18 cities + Unincorporated C...,2025,19,15636.0,15838.0,0.0,6237.0,13221.0,6298.0,3727.0,...,26111.0,0.23836,0.166266,0.281144,0.24611,0.37941,0.094302,0.23709,0.216015,0.663237


In [240]:
check = jurisdiction_year_dataset["bp_units_total"].sum() == sd_region_total["bp_units_total"].iloc[0]
print("Regional bp_units_total matches sum of jurisdiction rows:", check)

region_output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_regional_total.csv"
sd_region_total.to_csv(region_output_path, index=False)
print("Saved:", region_output_path)


Regional bp_units_total matches sum of jurisdiction rows: True
Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/rhna_housing_production_2025_regional_total.csv


## Power BI-ready export (long format)

This is the final long-format fact table for RHNA and APR production.

- APR activity is annual from 2018-2025.
- RHNA allocation/progress is a cumulative 6th Cycle snapshot and is therefore
  attached only to `TARGET_YEAR` rather than duplicated across earlier years.
- Jurisdiction and San Diego regional rows use the same column structure.
- Applications, entitlements, building permits, completions, and RHNA measures
  remain distinct development stages.
- Each metric includes an `income_category = "all"` row for the official
  headline total, plus separate income-tier rows for breakdowns.
- **Do not sum the `"all"` row together with the tier rows in Power BI.**
  For applications specifically, HCD's income-tier fields do not always
  reconcile exactly to `TOT_PROPOSED_UNITS`, so the `"all"` row is the
  authoritative overall application total.
- Row-level source, period, geography, unit, reporting status, and limitations
  are included for dashboard use and validation.


In [241]:
INCOME_TIERS = ["very_low", "low", "moderate", "above_moderate"]

REGION_NAME = "San Diego Region (18 cities + Unincorporated County)"

JURISDICTION_DISPLAY = {
    normalize_jurisdiction(city): city
    for city in SAN_DIEGO_CITIES
}
JURISDICTION_DISPLAY["unincorporated san diego county"] = (
    "Unincorporated San Diego County"
)

# ------------------------------------------------------------------
# APR production stages
#
# IMPORTANT:
# "all" is an explicit official total row. It is NOT calculated by
# summing the income-category rows in the Power BI export.
# ------------------------------------------------------------------

PRODUCTION_STAGE_CONFIG = {
    "Application": {
        "data": applications_history,
        "source": "HCD APR Table A",
        "categories": {
            "all": "application_units_total",
            "very_low": "application_very_low_total",
            "low": "application_low_total",
            "moderate": "application_moderate_total",
            "above_moderate": "application_above_moderate_total",
            "unclassified": "application_unclassified_total",
        },
        "limitations": (
            "The 'all' row uses HCD TOT_PROPOSED_UNITS and is the authoritative "
            "overall application total. Income-tier fields may not sum exactly "
            "to TOT_PROPOSED_UNITS; do not derive the overall total by summing "
            "tier rows. 'unclassified' is only the positive gap where the HCD "
            "overall total exceeds the reported tier sum."
        ),
    },
    "Entitlement": {
        "data": production_history,
        "source": "HCD APR Table A2",
        "categories": {
            "all": "ent_units_total",
            "very_low": "ent_very_low_total",
            "low": "ent_low_total",
            "moderate": "ent_moderate_total",
            "above_moderate": "ent_above_moderate_total",
        },
        "limitations": (
            "Self-reported by jurisdictions to HCD. The 'all' row is the "
            "headline total; do not sum it together with tier rows."
        ),
    },
    "Building Permit": {
        "data": production_history,
        "source": "HCD APR Table A2",
        "categories": {
            "all": "bp_units_total",
            "very_low": "bp_very_low_total",
            "low": "bp_low_total",
            "moderate": "bp_moderate_total",
            "above_moderate": "bp_above_moderate_total",
        },
        "limitations": (
            "Self-reported by jurisdictions to HCD. Annual APR permit activity "
            "is separate from cumulative RHNA progress. The 'all' row is the "
            "headline total; do not sum it together with tier rows."
        ),
    },
    "Completion": {
        "data": production_history,
        "source": "HCD APR Table A2",
        "categories": {
            "all": "co_units_total",
            "very_low": "co_very_low_total",
            "low": "co_low_total",
            "moderate": "co_moderate_total",
            "above_moderate": "co_above_moderate_total",
        },
        "limitations": (
            "Certificate-of-occupancy / completion units reported through APR. "
            "The 'all' row is the headline total; do not sum it together with "
            "tier rows."
        ),
    },
}

production_rows = []

for stage_label, config in PRODUCTION_STAGE_CONFIG.items():
    source_df = config["data"]

    for category, column in config["categories"].items():
        if column not in source_df.columns:
            raise KeyError(
                f"Expected column '{column}' for Power BI stage '{stage_label}' "
                f"was not found."
            )

        chunk = source_df[["jur_clean", "year", "reporting_status"]].copy()
        chunk["jurisdiction"] = chunk["jur_clean"].map(JURISDICTION_DISPLAY)
        chunk["development_stage"] = stage_label
        chunk["income_category"] = category
        chunk["value"] = source_df[column]
        chunk["source"] = config["source"]
        chunk["source_year"] = chunk["year"]
        chunk["download_date"] = DOWNLOAD_DATE
        chunk["geographic_level"] = "Jurisdiction"
        chunk["unit_of_measurement"] = "Housing units (count)"
        chunk["limitations"] = config["limitations"]

        production_rows.append(
            chunk[
                [
                    "jurisdiction",
                    "year",
                    "development_stage",
                    "income_category",
                    "value",
                    "reporting_status",
                    "source",
                    "source_year",
                    "download_date",
                    "geographic_level",
                    "unit_of_measurement",
                    "limitations",
                ]
            ]
        )

jurisdiction_production_long = pd.concat(
    production_rows,
    ignore_index=True,
)

# Regional APR rows use the same category definitions.
regional_production_long = (
    jurisdiction_production_long
    .groupby(
        ["year", "development_stage", "income_category", "source"],
        as_index=False,
        dropna=False,
    )
    .agg(
        value=("value", lambda values: values.sum(min_count=1)),
        jurisdictions_reporting=("value", "count"),
    )
)

regional_production_long["jurisdiction"] = REGION_NAME
regional_production_long["reporting_status"] = np.where(
    regional_production_long["jurisdictions_reporting"].eq(19),
    "complete",
    "partial",
)
regional_production_long["source_year"] = regional_production_long["year"]
regional_production_long["download_date"] = DOWNLOAD_DATE
regional_production_long["geographic_level"] = "Regional total"
regional_production_long["unit_of_measurement"] = "Housing units (count)"
regional_production_long["limitations"] = np.where(
    regional_production_long["jurisdictions_reporting"].eq(19),
    (
        "Sum of all 18 cities plus Unincorporated San Diego County. "
        "Use income_category='all' for the headline total; do not sum the "
        "'all' row together with tier rows."
    ),
    (
        "Partial regional total because one or more jurisdiction-year records "
        "are missing from the source. Use income_category='all' for the "
        "headline total; do not sum the 'all' row together with tier rows."
    ),
)

regional_production_long = regional_production_long[
    [
        "jurisdiction",
        "year",
        "development_stage",
        "income_category",
        "value",
        "reporting_status",
        "source",
        "source_year",
        "download_date",
        "geographic_level",
        "unit_of_measurement",
        "limitations",
    ]
]

# ------------------------------------------------------------------
# RHNA cumulative snapshot
# ------------------------------------------------------------------

RHNA_STAGE_CONFIG = {
    "RHNA Allocation": {
        "categories": {
            "all": "rhna_target_total",
            "very_low": "rhna_target_very_low",
            "low": "rhna_target_low",
            "moderate": "rhna_target_moderate",
            "above_moderate": "rhna_target_above_moderate",
        },
        "unit": "Housing units (count)",
        "source_year": "6th Cycle 2021-2029 adopted allocation",
        "limitations": (
            "Adopted 6th Cycle allocation; not an annual production metric. "
            "Use income_category='all' for the headline total."
        ),
    },
    "RHNA Progress (Building Permits)": {
        "categories": {
            "all": "rhna_units_reported_total",
            "very_low": "rhna_reported_very_low",
            "low": "rhna_reported_low",
            "moderate": "rhna_reported_moderate",
            "above_moderate": "rhna_reported_above_moderate",
        },
        "unit": "Housing units (count)",
        "source_year": f"6th Cycle cumulative to date; pulled {TARGET_YEAR}",
        "limitations": (
            "Cumulative RHNA progress based on qualifying building permits "
            "issued, not completions or entitlements. Use income_category='all' "
            "for the headline total."
        ),
    },
    "RHNA Remaining": {
        "categories": {
            "all": "rhna_remaining_total",
            "very_low": "rhna_remaining_very_low",
            "low": "rhna_remaining_low",
            "moderate": "rhna_remaining_moderate",
            "above_moderate": "rhna_remaining_above_moderate",
        },
        "unit": "Housing units (count)",
        "source_year": f"6th Cycle cumulative to date; pulled {TARGET_YEAR}",
        "limitations": (
            "Allocation minus cumulative qualifying RHNA units, clipped at zero "
            "separately by income tier. Use income_category='all' for the "
            "headline total."
        ),
    },
    "RHNA Percent Complete": {
        "categories": {
            "all": "rhna_pct_achieved",
            "very_low": "rhna_pct_achieved_very_low",
            "low": "rhna_pct_achieved_low",
            "moderate": "rhna_pct_achieved_moderate",
            "above_moderate": "rhna_pct_achieved_above_moderate",
        },
        "unit": "Percent (0-1 share)",
        "source_year": f"6th Cycle cumulative to date; pulled {TARGET_YEAR}",
        "limitations": (
            "Cumulative RHNA progress ratio; not an annual production rate. "
            "Use income_category='all' for the headline total."
        ),
    },
}

rhna_rows = []

for stage_label, config in RHNA_STAGE_CONFIG.items():
    for category, value_column in config["categories"].items():
        if value_column not in rhna_summary.columns:
            raise KeyError(
                f"Expected RHNA column '{value_column}' for '{stage_label}' "
                f"was not found."
            )

        chunk = rhna_summary[["jur_clean", value_column]].copy()
        chunk["jurisdiction"] = chunk["jur_clean"].map(JURISDICTION_DISPLAY)
        chunk["year"] = TARGET_YEAR
        chunk["development_stage"] = stage_label
        chunk["income_category"] = category
        chunk["value"] = chunk[value_column]
        chunk["reporting_status"] = "reported"
        chunk["source"] = "HCD RHNA 6th Cycle Progress Report"
        chunk["source_year"] = config["source_year"]
        chunk["download_date"] = DOWNLOAD_DATE
        chunk["geographic_level"] = "Jurisdiction"
        chunk["unit_of_measurement"] = config["unit"]
        chunk["limitations"] = config["limitations"]

        rhna_rows.append(
            chunk[
                [
                    "jurisdiction",
                    "year",
                    "development_stage",
                    "income_category",
                    "value",
                    "reporting_status",
                    "source",
                    "source_year",
                    "download_date",
                    "geographic_level",
                    "unit_of_measurement",
                    "limitations",
                ]
            ]
        )

jurisdiction_rhna_long = pd.concat(
    rhna_rows,
    ignore_index=True,
)

regional_rhna_rows = []

for stage_label, config in RHNA_STAGE_CONFIG.items():
    for category, value_column in config["categories"].items():
        if value_column not in sd_region_total.columns:
            raise KeyError(
                f"Expected regional RHNA column '{value_column}' for "
                f"'{stage_label}' was not found."
            )

        regional_rhna_rows.append(
            {
                "jurisdiction": REGION_NAME,
                "year": TARGET_YEAR,
                "development_stage": stage_label,
                "income_category": category,
                "value": sd_region_total[value_column].iloc[0],
                "reporting_status": "complete",
                "source": "HCD RHNA 6th Cycle Progress Report",
                "source_year": config["source_year"],
                "download_date": DOWNLOAD_DATE,
                "geographic_level": "Regional total",
                "unit_of_measurement": config["unit"],
                "limitations": config["limitations"],
            }
        )

regional_rhna_long = pd.DataFrame(regional_rhna_rows)

powerbi_long = pd.concat(
    [
        jurisdiction_production_long,
        regional_production_long,
        jurisdiction_rhna_long,
        regional_rhna_long,
    ],
    ignore_index=True,
)

powerbi_long = (
    powerbi_long
    .sort_values(
        ["jurisdiction", "year", "development_stage", "income_category"]
    )
    .reset_index(drop=True)
)

print(powerbi_long.shape)
print("Years present:", sorted(powerbi_long["year"].dropna().unique()))
powerbi_long.head(20)


(3760, 12)
Years present: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,jurisdiction,year,development_stage,income_category,value,reporting_status,source,source_year,download_date,geographic_level,unit_of_measurement,limitations
0,Carlsbad,2018,Application,above_moderate,313.0,reported,HCD APR Table A,2018,2026-08-17,Jurisdiction,Housing units (count),The 'all' row uses HCD TOT_PROPOSED_UNITS and ...
1,Carlsbad,2018,Application,all,369.0,reported,HCD APR Table A,2018,2026-08-17,Jurisdiction,Housing units (count),The 'all' row uses HCD TOT_PROPOSED_UNITS and ...
2,Carlsbad,2018,Application,low,43.0,reported,HCD APR Table A,2018,2026-08-17,Jurisdiction,Housing units (count),The 'all' row uses HCD TOT_PROPOSED_UNITS and ...
3,Carlsbad,2018,Application,moderate,6.0,reported,HCD APR Table A,2018,2026-08-17,Jurisdiction,Housing units (count),The 'all' row uses HCD TOT_PROPOSED_UNITS and ...
4,Carlsbad,2018,Application,unclassified,0.0,reported,HCD APR Table A,2018,2026-08-17,Jurisdiction,Housing units (count),The 'all' row uses HCD TOT_PROPOSED_UNITS and ...
5,Carlsbad,2018,Application,very_low,7.0,reported,HCD APR Table A,2018,2026-08-17,Jurisdiction,Housing units (count),The 'all' row uses HCD TOT_PROPOSED_UNITS and ...
6,Carlsbad,2018,Building Permit,above_moderate,210.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),Self-reported by jurisdictions to HCD. Annual ...
7,Carlsbad,2018,Building Permit,all,243.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),Self-reported by jurisdictions to HCD. Annual ...
8,Carlsbad,2018,Building Permit,low,5.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),Self-reported by jurisdictions to HCD. Annual ...
9,Carlsbad,2018,Building Permit,moderate,28.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),Self-reported by jurisdictions to HCD. Annual ...


In [242]:
required_powerbi_columns = [
    "jurisdiction",
    "year",
    "development_stage",
    "income_category",
    "value",
    "reporting_status",
    "source",
    "source_year",
    "download_date",
    "geographic_level",
    "unit_of_measurement",
    "limitations",
]

assert set(required_powerbi_columns).issubset(powerbi_long.columns)

duplicate_keys = powerbi_long.duplicated(
    subset=["jurisdiction", "year", "development_stage", "income_category"]
)
assert not duplicate_keys.any(), "Duplicate Power BI fact-table keys found."


def powerbi_jurisdiction_total(stage_label):
    """Return target-year jurisdiction rows for the official 'all' category."""
    result = (
        powerbi_long[
            (powerbi_long["year"] == TARGET_YEAR)
            & (powerbi_long["development_stage"] == stage_label)
            & (powerbi_long["geographic_level"] == "Jurisdiction")
            & (powerbi_long["income_category"] == "all")
        ]
        .set_index("jurisdiction")["value"]
    )

    assert result.index.is_unique, (
        f"Duplicate jurisdiction total rows found for {stage_label}."
    )

    return result


# ------------------------------------------------------------------
# APR production totals
# Validate the explicit "all" rows against the source totals.
# Do NOT validate by summing tier rows, because HCD application tier
# fields do not always reconcile exactly to TOT_PROPOSED_UNITS.
# ------------------------------------------------------------------

production_total_checks = [
    (
        "Application",
        applications_history,
        "application_units_total",
    ),
    (
        "Entitlement",
        production_history,
        "ent_units_total",
    ),
    (
        "Building Permit",
        production_history,
        "bp_units_total",
    ),
    (
        "Completion",
        production_history,
        "co_units_total",
    ),
]

for stage_label, source_df, source_column in production_total_checks:
    actual = powerbi_jurisdiction_total(stage_label)

    expected = (
        source_df[source_df["year"].eq(TARGET_YEAR)]
        .assign(
            jurisdiction=lambda df: df["jur_clean"].map(
                JURISDICTION_DISPLAY
            )
        )
        .set_index("jurisdiction")[source_column]
    )

    assert np.allclose(
        actual.reindex(expected.index),
        expected,
        equal_nan=True,
    ), (
        f"Power BI '{stage_label}' all-income totals do not match "
        f"source column '{source_column}'."
    )


# ------------------------------------------------------------------
# Application tier rows are validated separately against the HCD
# tier-classified total.
#
# This intentionally does NOT compare the tier sum to TOT_PROPOSED_UNITS.
# In the source data, some jurisdiction-years have tier sums above or
# below HCD's overall proposed-unit total.
# ------------------------------------------------------------------

application_tier_check = (
    powerbi_long[
        (powerbi_long["year"] == TARGET_YEAR)
        & (powerbi_long["development_stage"] == "Application")
        & (powerbi_long["geographic_level"] == "Jurisdiction")
        & (powerbi_long["income_category"].isin(INCOME_TIERS))
    ]
    .groupby("jurisdiction")["value"]
    .sum(min_count=1)
)

expected_application_classified = (
    applications_history[
        applications_history["year"].eq(TARGET_YEAR)
    ]
    .assign(
        jurisdiction=lambda df: df["jur_clean"].map(
            JURISDICTION_DISPLAY
        )
    )
    .set_index("jurisdiction")["application_classified_total"]
)

assert np.allclose(
    application_tier_check.reindex(
        expected_application_classified.index
    ),
    expected_application_classified,
    equal_nan=True,
), "Power BI application tier rows do not match HCD tier-classified totals."


application_unclassified_check = (
    powerbi_long[
        (powerbi_long["year"] == TARGET_YEAR)
        & (powerbi_long["development_stage"] == "Application")
        & (powerbi_long["geographic_level"] == "Jurisdiction")
        & (powerbi_long["income_category"] == "unclassified")
    ]
    .set_index("jurisdiction")["value"]
)

expected_application_unclassified = (
    applications_history[
        applications_history["year"].eq(TARGET_YEAR)
    ]
    .assign(
        jurisdiction=lambda df: df["jur_clean"].map(
            JURISDICTION_DISPLAY
        )
    )
    .set_index("jurisdiction")["application_unclassified_total"]
)

assert np.allclose(
    application_unclassified_check.reindex(
        expected_application_unclassified.index
    ),
    expected_application_unclassified,
    equal_nan=True,
), "Power BI application unclassified rows do not match the processed source."


# ------------------------------------------------------------------
# RHNA "all" rows: validate jurisdiction totals and regional totals.
# ------------------------------------------------------------------

rhna_total_columns = {
    "RHNA Allocation": "rhna_target_total",
    "RHNA Progress (Building Permits)": "rhna_units_reported_total",
    "RHNA Remaining": "rhna_remaining_total",
    "RHNA Percent Complete": "rhna_pct_achieved",
}

for stage_label, source_column in rhna_total_columns.items():
    actual_jurisdiction = powerbi_jurisdiction_total(stage_label)

    expected_jurisdiction = (
        rhna_summary
        .assign(
            jurisdiction=lambda df: df["jur_clean"].map(
                JURISDICTION_DISPLAY
            )
        )
        .set_index("jurisdiction")[source_column]
    )

    assert np.allclose(
        actual_jurisdiction.reindex(expected_jurisdiction.index),
        expected_jurisdiction,
        equal_nan=True,
    ), (
        f"Power BI '{stage_label}' jurisdiction totals do not match "
        f"'{source_column}'."
    )

    actual_regional_rows = powerbi_long[
        (powerbi_long["year"] == TARGET_YEAR)
        & (powerbi_long["development_stage"] == stage_label)
        & (powerbi_long["geographic_level"] == "Regional total")
        & (powerbi_long["income_category"] == "all")
    ]

    assert len(actual_regional_rows) == 1, (
        f"Expected exactly one regional all-income row for {stage_label}."
    )

    actual_regional = actual_regional_rows["value"].iloc[0]
    expected_regional = sd_region_total[source_column].iloc[0]

    assert np.isclose(
        actual_regional,
        expected_regional,
        equal_nan=True,
    ), (
        f"Power BI '{stage_label}' regional total does not match "
        f"'{source_column}'."
    )


print("Power BI long-format structural checks passed.")
print("Rows:", len(powerbi_long))

powerbi_path = (
    PROCESSED_DIR
    / f"powerbi_rhna_production_{APR_START_YEAR}_{TARGET_YEAR}_long.csv"
)

powerbi_long.to_csv(powerbi_path, index=False)

print("Saved:", powerbi_path)


Power BI long-format structural checks passed.
Rows: 3760
Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/powerbi_rhna_production_2018_2025_long.csv


**Coverage:** spans 2018-2025 for all 4 APR stages. Check the year-coverage
output above (applications and permits/completions sections) for any year
with fewer than 19 jurisdictions before filtering to it in Power BI.

**Power BI total rule:** use `income_category = "all"` for headline totals.
The `"all"` row is already the official total and must not be summed together
with the tier rows. This is especially important for applications because
HCD's income-tier fields can differ from `TOT_PROPOSED_UNITS`.


## Metric dictionary

In [243]:
metric_dictionary = pd.DataFrame([
    {
        "output_metric": "application_units_total / application_<tier>_total",
        "source_table": "HCD APR Table A",
        "source_variables": "TOT_PROPOSED_UNITS",
        "definition": "Total proposed housing units in applications submitted, per jurisdiction-year. All application statuses are included; this is submitted activity, not an approval measure.",
        "source_year": f"{APR_START_YEAR}-{TARGET_YEAR}",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Use the HCD TOT_PROPOSED_UNITS-based all-income total for headline application counts. Income-tier fields can sum above or below that overall total and should not be used to reconstruct it. application_unclassified_total retains only a positive overall-minus-tier gap.",
    },
    {
        "output_metric": "application_unclassified_total",
        "source_table": "HCD APR Table A",
        "source_variables": "TOT_PROPOSED_UNITS minus sum of income-tier unit fields",
        "definition": "Positive gap where HCD TOT_PROPOSED_UNITS exceeds the sum of reported application income-tier units.",
        "source_year": f"{APR_START_YEAR}-{TARGET_YEAR}",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Clipped at zero. If the reported tier sum exceeds TOT_PROPOSED_UNITS, this field remains zero; use the all-income total as the authoritative overall application count.",
    },
    {
        "output_metric": "ent_units_total / ent_<tier>_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of unprefixed *_INCOME_* columns (not BP_ or CO_ prefixed)",
        "definition": "Total housing units with a planning entitlement approved, all income tiers, per jurisdiction-year. Kept fully separate from bp_units_total and co_units_total - never summed together",
        "source_year": f"2018-{TARGET_YEAR} (historical export); {TARGET_YEAR} (jurisdiction-year table)", "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Matches HCD's own auto-populated NO_ENTITLEMENTS total exactly for all 19 jurisdictions in the target-year validation",
    },
    {
        "output_metric": "bp_units_total / bp_<tier>_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of BP_*_INCOME columns, kept both as an overall total and broken out per income tier (very_low, low, moderate, above_moderate)",
        "definition": "Total housing units with a building permit issued, per jurisdiction-year. Per-tier columns sum back to the overall total exactly (enforced by an assertion in the notebook)",
        "source_year": f"2018-{TARGET_YEAR} (historical export); {TARGET_YEAR} (jurisdiction-year table)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Self-reported by jurisdictions to HCD, not independently verified",
    },
    {
        "output_metric": "co_units_total / co_<tier>_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of CO_*_INCOME columns, kept both as an overall total and broken out per income tier (very_low, low, moderate, above_moderate)",
        "definition": "Total housing units with a certificate of occupancy (completed), per jurisdiction-year. Per-tier columns sum back to the overall total exactly (enforced by an assertion in the notebook)",
        "source_year": f"2018-{TARGET_YEAR} (historical export); {TARGET_YEAR} (jurisdiction-year table)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Self-reported by jurisdictions to HCD, not independently verified",
    },
    {
        "output_metric": "bp_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "bp_affordable_total / bp_units_total",
        "definition": "Share of permitted units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Percent (0-1 share)",
        "limitations": "NaN when bp_units_total is 0 that year - zero permits, not missing data",
    },
    {
        "output_metric": "co_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "co_affordable_total / co_units_total",
        "definition": "Share of completed units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Percent (0-1 share)",
        "limitations": "NaN when co_units_total is 0 that year - zero completions, not missing data",
    },
    {
        "output_metric": "rhna_target_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD",
        "definition": "Assigned RHNA target units, kept separate per income tier (very_low, low, moderate, above_moderate) for the 6th Cycle planning period, per jurisdiction - plus rhna_target_total for the summed value",
        "source_year": "6th Cycle (2021-2029), adopted allocation, does not change during the cycle",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county) and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Validated exact match against HCD/SANDAG published figures",
    },
    {
        "output_metric": "rhna_reported_<tier> / rhna_pct_achieved_<tier> / rhna_remaining_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS",
        "definition": "BASED ON BUILDING PERMITS ISSUED, not completed units - this is HCD's own RHNA-credit methodology. Kept separate per income tier, per jurisdiction. Cumulative cycle-to-date, not year-by-year. Do not conflate with co_units_total (completions)",
        "source_year": f"Cumulative, 6th Cycle to date (as of {TARGET_YEAR} data pull)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county) and regional total",
        "unit_of_measurement": "Housing units (count); Percent for rhna_pct_achieved_<tier>",
        "limitations": "Cumulative cycle-to-date, not annual",
    },
    {
        "output_metric": "housing_units_total",
        "source_table": "California DOF E-5 (2026 release; revised 1/1/2025 estimate)",
        "source_variables": "Total housing units",
        "definition": "Estimated total existing housing stock as of January 1.",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Annual point-in-time estimate; not annual housing production.",
    },
    {
        "output_metric": "occupied_units",
        "source_table": "California DOF E-5 (2026 release; revised 1/1/2025 estimate)",
        "source_variables": "Occupied",
        "definition": "Estimated occupied housing units.",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Annual January 1 estimate.",
    },
    {
        "output_metric": "vacant_units",
        "source_table": "California DOF E-5 (2026 release; revised 1/1/2025 estimate)",
        "source_variables": "housing_units_total - Occupied",
        "definition": "Estimated vacant housing units.",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Derived from DOF E-5 estimates.",
    },
    {
        "output_metric": "single_family_units",
        "source_table": "California DOF E-5 (2026 release; revised 1/1/2025 estimate)",
        "source_variables": "Single Detached + Single Attached",
        "definition": "Estimated single-family housing units.",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Combines detached and attached single-family categories.",
    },
    {
        "output_metric": "multifamily_units",
        "source_table": "California DOF E-5 (2026 release; revised 1/1/2025 estimate)",
        "source_variables": "Two to Four + Five Plus",
        "definition": "Estimated multifamily housing units.",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Combines 2-4-unit and 5+-unit categories.",
    },
    {
        "output_metric": "mobile_home_units",
        "source_table": "California DOF E-5 (2026 release; revised 1/1/2025 estimate)",
        "source_variables": "Mobile Homes",
        "definition": "Estimated mobile-home units.",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Annual January 1 estimate.",
    },
    {
        "output_metric": "sd_permit_du_by_tier",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_DU_EXTREMELY_LOW/VERY_LOW/LOW/MODERATE/ABOVE_MODERATE",
        "definition": "Dwelling units per approval, broken out by income tier, City of San Diego only. Separate ADU (APPROVAL_ADU_*) and JADU (APPROVAL_JADU_*) columns exist alongside standard units",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "City of San Diego only",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Overlaps with APR for San Diego - do not sum with bp_units_total. Appendix data, not in core exports",
    },
    {
        "output_metric": "sd_permit_stage",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_ISSUE_DATE, APPROVAL_CLOSE_DATE, approval_status",
        "definition": "This dataset tracks permit issuance and closure only - it has no certificate-of-occupancy / completion field equivalent to APR's CO_* columns",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "City of San Diego only",
        "unit_of_measurement": "Date / status (not a count)",
        "limitations": "'Closed' can mean finaled, expired, or withdrawn - not necessarily completed construction",
    },
])
metric_dictionary.to_csv(DOCS_DIR / "rhna_housing_production_metric_dictionary.csv", index=False)
metric_dictionary


,output_metric,source_table,source_variables,definition,source_year,download_date,geographic_level,unit_of_measurement,limitations
0,application_units_total / application_<tier>_t...,HCD APR Table A,TOT_PROPOSED_UNITS,Total proposed housing units in applications s...,2018-2025,2026-08-17,Jurisdiction and regional total,Housing units (count),Use the HCD TOT_PROPOSED_UNITS-based all-incom...
1,application_unclassified_total,HCD APR Table A,TOT_PROPOSED_UNITS minus sum of income-tier un...,Positive gap where HCD TOT_PROPOSED_UNITS exce...,2018-2025,2026-08-17,Jurisdiction and regional total,Housing units (count),Clipped at zero. If the reported tier sum exce...
2,ent_units_total / ent_<tier>_total,HCD APR Table A2,Sum of unprefixed *_INCOME_* columns (not BP_ ...,Total housing units with a planning entitlemen...,2018-2025 (historical export); 2025 (jurisdict...,2026-08-17,Jurisdiction (18 cities + unincorporated county),Housing units (count),Matches HCD's own auto-populated NO_ENTITLEMEN...
3,bp_units_total / bp_<tier>_total,HCD APR Table A2,"Sum of BP_*_INCOME columns, kept both as an ov...",Total housing units with a building permit iss...,2018-2025 (historical export); 2025 (jurisdict...,2026-08-17,Jurisdiction (18 cities + unincorporated county),Housing units (count),"Self-reported by jurisdictions to HCD, not ind..."
4,co_units_total / co_<tier>_total,HCD APR Table A2,"Sum of CO_*_INCOME columns, kept both as an ov...",Total housing units with a certificate of occu...,2018-2025 (historical export); 2025 (jurisdict...,2026-08-17,Jurisdiction (18 cities + unincorporated county),Housing units (count),"Self-reported by jurisdictions to HCD, not ind..."
5,bp_affordable_share,HCD APR Table A2,bp_affordable_total / bp_units_total,"Share of permitted units in VLI, LI, or Modera...",2025,2026-08-17,Jurisdiction (18 cities + unincorporated county),Percent (0-1 share),NaN when bp_units_total is 0 that year - zero ...
6,co_affordable_share,HCD APR Table A2,co_affordable_total / co_units_total,"Share of completed units in VLI, LI, or Modera...",2025,2026-08-17,Jurisdiction (18 cities + unincorporated county),Percent (0-1 share),NaN when co_units_total is 0 that year - zero ...
7,rhna_target_<tier>,HCD RHNA 6th Cycle Progress Report (Table B),"RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD","Assigned RHNA target units, kept separate per ...","6th Cycle (2021-2029), adopted allocation, doe...",2026-08-17,Jurisdiction (18 cities + unincorporated count...,Housing units (count),Validated exact match against HCD/SANDAG publi...
8,rhna_reported_<tier> / rhna_pct_achieved_<tier...,HCD RHNA 6th Cycle Progress Report (Table B),"VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS","BASED ON BUILDING PERMITS ISSUED, not complete...","Cumulative, 6th Cycle to date (as of 2025 data...",2026-08-17,Jurisdiction (18 cities + unincorporated count...,Housing units (count); Percent for rhna_pct_ac...,"Cumulative cycle-to-date, not annual"
9,housing_units_total,California DOF E-5 (2026 release; revised 1/1/...,Total housing units,Estimated total existing housing stock as of J...,2025,2026-08-17,Jurisdiction and regional total,Housing units (count),Annual point-in-time estimate; not annual hous...


## Export

In [244]:
output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv"
jurisdiction_year_dataset.to_csv(output_path, index=False)
print("Saved:", output_path)


Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/rhna_housing_production_2025_by_jurisdiction.csv


### Export metadata

Dataset-level metadata for each exported file - kept as its own record
rather than repeated on every row.

In [245]:
export_metadata = pd.DataFrame([
    {
        "file_name": f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County)",
        "unit_of_measurement": "Housing units (count); percent for share/pct_achieved fields",
        "limitations": "See data_quality_limitations.csv for full detail",
    },
    {
        "file_name": f"apr_production_{APR_START_YEAR}_{TARGET_YEAR}_by_jurisdiction_year.csv",
        "source_year": f"{APR_START_YEAR}-{TARGET_YEAR}",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County), by year",
        "unit_of_measurement": "Housing units (count); percent for affordable-share fields",
        "limitations": "2018 (APR's first collection year) may have lower reporting completeness than later years",
    },
    {
        "file_name": f"rhna_housing_production_{TARGET_YEAR}_regional_total.csv",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Regional (all 18 cities + unincorporated San Diego County combined)",
        "unit_of_measurement": "Housing units (count); percent for share/pct_achieved fields",
        "limitations": "Percent fields recalculated at the regional level, not averaged from jurisdiction percentages",
    },
    {
        "file_name": f"powerbi_rhna_production_{APR_START_YEAR}_{TARGET_YEAR}_long.csv",
        "source_year": f"{APR_START_YEAR}-{TARGET_YEAR} plus current RHNA 6th Cycle snapshot",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction and regional total",
        "unit_of_measurement": "Housing units (count) or percent, identified per row",
        "limitations": "APR rows are annual 2018-2025; RHNA rows are a cumulative 6th Cycle snapshot attached to TARGET_YEAR. Row-level metadata and reporting status are included.",
    },
])
export_metadata.to_csv(DOCS_DIR / "export_metadata.csv", index=False)
export_metadata


,file_name,source_year,download_date,geographic_level,unit_of_measurement,limitations
0,rhna_housing_production_2025_by_jurisdiction.csv,2025,2026-08-17,Jurisdiction (18 cities + unincorporated San D...,Housing units (count); percent for share/pct_a...,See data_quality_limitations.csv for full detail
1,apr_production_2018_2025_by_jurisdiction_year.csv,2018-2025,2026-08-17,Jurisdiction (18 cities + unincorporated San D...,Housing units (count); percent for affordable-...,2018 (APR's first collection year) may have lo...
2,rhna_housing_production_2025_regional_total.csv,2025,2026-08-17,Regional (all 18 cities + unincorporated San D...,Housing units (count); percent for share/pct_a...,Percent fields recalculated at the regional le...
3,powerbi_rhna_production_2018_2025_long.csv,2018-2025 plus current RHNA 6th Cycle snapshot,2026-08-17,Jurisdiction and regional total,"Housing units (count) or percent, identified p...",APR rows are annual 2018-2025; RHNA rows are a...


## Validate against HCD's published RHNA results

Checks `rhna_target_total` against figures from official adopted planning
documents (SANDAG Board Resolution, adopted Housing Elements) - a
different publication channel than the CKAN dataset this notebook pulls
from.

In [246]:
PUBLISHED_RHNA = pd.DataFrame([
    {
        "jur_clean": "san diego",
        "published_rhna_target": 108036,
        "source": "City of San Diego adopted Housing Element 2021-2029",
    },
    {
        "jur_clean": "unincorporated san diego county",
        "published_rhna_target": 6700,
        "source": "County of San Diego General Plan Annual Progress Report",
    },
    {
        "jur_clean": "oceanside",
        "published_rhna_target": 5443,
        "source": "City of Oceanside Housing Element page",
    },
])

rhna_check = jurisdiction_year_dataset[
    ["jur_clean", "rhna_target_total"]
].merge(
    PUBLISHED_RHNA,
    on="jur_clean",
    how="inner",
)

rhna_check["diff"] = (
    rhna_check["rhna_target_total"]
    - rhna_check["published_rhna_target"]
)

assert rhna_check["diff"].eq(0).all(), (
    "One or more jurisdiction RHNA allocation totals do not match "
    "the separately published values."
)

print("Jurisdiction-level adopted RHNA allocation checks:")
display(rhna_check)

regional_published = 171685
regional_ours = sd_region_total["rhna_target_total"].iloc[0]

assert regional_ours == regional_published, (
    "Regional RHNA allocation does not match the published regional total."
)

print(
    f"Regional total: ours={regional_ours:.0f}, "
    f"published={regional_published}, match=True"
)

# Transformation check for allocation/progress/remaining/percent by tier.
# This is not a separate external source; it confirms that the processed
# RHNA fields preserve and correctly calculate the HCD source fields.
RHNA_VALIDATION_JURISDICTIONS = [
    "san diego",
    "oceanside",
    "unincorporated san diego county",
]

rhna_transform_checks = []

for jurisdiction in RHNA_VALIDATION_JURISDICTIONS:
    source_row = sd_rhna6[
        sd_rhna6["jur_clean"].eq(jurisdiction)
    ].iloc[0]

    processed_row = rhna_summary[
        rhna_summary["jur_clean"].eq(jurisdiction)
    ].iloc[0]

    for tier, (target_col, reported_col) in RHNA_TIERS.items():
        source_target = source_row[target_col]
        source_reported = source_row[reported_col]
        expected_remaining = max(source_target - source_reported, 0)
        expected_pct = (
            source_reported / source_target
            if source_target != 0
            else np.nan
        )

        checks = {
            "allocation": (
                source_target,
                processed_row[f"rhna_target_{tier}"],
            ),
            "qualifying_units_reported": (
                source_reported,
                processed_row[f"rhna_reported_{tier}"],
            ),
            "remaining": (
                expected_remaining,
                processed_row[f"rhna_remaining_{tier}"],
            ),
            "pct_complete": (
                expected_pct,
                processed_row[f"rhna_pct_achieved_{tier}"],
            ),
        }

        for metric, (expected_value, processed_value) in checks.items():
            rhna_transform_checks.append(
                {
                    "jurisdiction": jurisdiction,
                    "income_category": tier,
                    "metric": metric,
                    "expected_value": expected_value,
                    "processed_value": processed_value,
                    "match": np.isclose(
                        expected_value,
                        processed_value,
                        equal_nan=True,
                    ),
                }
            )

rhna_transform_validation = pd.DataFrame(rhna_transform_checks)

assert rhna_transform_validation["match"].all(), (
    "RHNA tier-level transformation validation failed."
)

display(rhna_transform_validation)


Jurisdiction-level adopted RHNA allocation checks:


,jur_clean,rhna_target_total,published_rhna_target,source,diff
0,oceanside,5443,5443,City of Oceanside Housing Element page,0
1,san diego,108036,108036,City of San Diego adopted Housing Element 2021...,0
2,unincorporated san diego county,6700,6700,County of San Diego General Plan Annual Progre...,0


Regional total: ours=171685, published=171685, match=True


,jurisdiction,income_category,metric,expected_value,processed_value,match
0,san diego,very_low,allocation,27549.000000,27549.000000,True
1,san diego,very_low,qualifying_units_reported,2513.000000,2513.000000,True
2,san diego,very_low,remaining,25036.000000,25036.000000,True
3,san diego,very_low,pct_complete,0.091219,0.091219,True
4,san diego,low,allocation,17331.000000,17331.000000,True
5,san diego,low,qualifying_units_reported,2958.000000,2958.000000,True
6,san diego,low,remaining,14373.000000,14373.000000,True
7,san diego,low,pct_complete,0.170677,0.170677,True
8,san diego,moderate,allocation,19319.000000,19319.000000,True
9,san diego,moderate,qualifying_units_reported,1360.000000,1360.000000,True


**Validation result:** the adopted RHNA allocation total is checked against
separately published planning figures for San Diego, Oceanside, Unincorporated
San Diego County, and the San Diego regional total.

The additional tier-level check confirms that allocation, qualifying units
reported, remaining need, and percent complete are transformed correctly from
the HCD RHNA source. This transformation check is not treated as an independent
external validation source.

APR entitlement, building-permit, and completion checks are separate and are
not used as validation of RHNA progress.


## Validation against dashboard prototype

In [247]:
def find_sibling_repo(repo_name: str, search_depth: int = 3) -> Path | None:
    candidates = [ROOT, *ROOT.parents][:search_depth + 2]
    for base in candidates:
        match = base / repo_name
        if match.exists():
            return match
        if base.parent.exists():
            for sibling in base.parent.iterdir():
                if sibling.name == repo_name and sibling.is_dir():
                    return sibling
    return None


baseline_repo = find_sibling_repo("housing-dashboard-prototype")

if baseline_repo is None:
    print(
        "Could not find a local clone of housing-dashboard-prototype near this repo. "
        "Clone it into the same parent folder as this repo, then re-run this cell."
    )
else:
    BASELINE_PATH = baseline_repo / "data" / "processed" / "sd_apr_a2_city_year_supply.csv"
    print("Found baseline repo at:", baseline_repo)


Found baseline repo at: /Users/laurenvo/Documents/Github/housing-dashboard-prototype


In [248]:
if baseline_repo is not None and BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
    baseline["jur_clean"] = baseline["jur_clean"].str.lower()

    baseline_years = sorted(baseline["year"].unique())
    our_years = sorted(production_history["year"].dropna().unique())
    overlapping_years = sorted(set(baseline_years) & set(our_years))
    years_ahead = sorted(set(our_years) - set(baseline_years))

    if years_ahead:
        print(
            f"Baseline (dashboard prototype) does not yet include: {years_ahead}. "
            "Those years are excluded from the prototype comparison."
        )

    if not overlapping_years:
        print(
            "No overlapping years between this notebook's pull and the "
            "baseline - nothing to validate."
        )
    else:
        comparison = production_history.merge(
            baseline,
            on=["jur_clean", "year"],
            suffixes=("_fresh", "_baseline"),
            how="inner",
        )

        assert not comparison.duplicated(
            subset=["jur_clean", "year"]
        ).any(), (
            "Validation merge created duplicate jurisdiction-year rows. "
            "The comparison must be keyed on both jurisdiction and year."
        )

        comparison["bp_difference"] = (
            comparison["bp_units_total_fresh"]
            - comparison["bp_units_total_baseline"]
        )

        comparison["bp_matches"] = comparison["bp_difference"].eq(0)
        comparison["validation_status"] = np.where(
            comparison["bp_matches"],
            "match",
            "fresh HCD pull differs from older dashboard snapshot",
        )

        failed_checks = comparison[~comparison["bp_matches"]].copy()

        print(f"Validated years: {overlapping_years}")
        print(
            f"Compared {len(comparison)} jurisdiction-year rows; "
            f"{comparison['bp_matches'].sum()} exact matches; "
            f"{len(failed_checks)} documented source-version differences."
        )

        display(
            failed_checks[
                [
                    "jur_clean",
                    "year",
                    "bp_units_total_fresh",
                    "bp_units_total_baseline",
                    "bp_difference",
                    "validation_status",
                ]
            ]
        )

        comparison.to_csv(
            PROCESSED_DIR
            / "rhna_housing_production_validation_report.csv",
            index=False,
        )

elif baseline_repo is not None:
    print(f"Repo found but expected file is missing: {BASELINE_PATH}")


Baseline (dashboard prototype) does not yet include: [np.int64(2025)]. Those years are excluded from the prototype comparison.
Validated years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Compared 126 jurisdiction-year rows; 103 exact matches; 23 documented source-version differences.


,jur_clean,year,bp_units_total_fresh,bp_units_total_baseline,bp_difference,validation_status
7,chula vista,2018,1777,1675.0,102.0,fresh HCD pull differs from older dashboard sn...
8,chula vista,2019,840,2126.0,-1286.0,fresh HCD pull differs from older dashboard sn...
10,chula vista,2021,1756,1852.0,-96.0,fresh HCD pull differs from older dashboard sn...
11,chula vista,2022,1178,1182.0,-4.0,fresh HCD pull differs from older dashboard sn...
16,coronado,2020,29,51.0,-22.0,fresh HCD pull differs from older dashboard sn...
30,el cajon,2020,41,45.0,-4.0,fresh HCD pull differs from older dashboard sn...
33,el cajon,2023,139,140.0,-1.0,fresh HCD pull differs from older dashboard sn...
38,encinitas,2021,153,149.0,4.0,fresh HCD pull differs from older dashboard sn...
39,encinitas,2022,157,152.0,5.0,fresh HCD pull differs from older dashboard sn...
40,encinitas,2023,279,269.0,10.0,fresh HCD pull differs from older dashboard sn...


## Appendix: City of San Diego permits (optional)

Not part of the core pipeline - doesn't feed jurisdiction_year_dataset or
any export. Used only to validate APR's San Diego permit numbers against
the City's own system (overlap check below).

In [249]:
permits_dir = RAW_DIR.parent / "sandiego_permits"
permits_dir.mkdir(parents=True, exist_ok=True)

CITY_PERMITS_OPTIONAL = True
permits_frames = []

for label in ["active", "closed"]:
    raw_path = permits_dir / f"{label}_approvals_raw.csv"

    if not raw_path.exists():
        print(f"Optional City of SD permit file not found: {raw_path}")
        continue

    df = pd.read_csv(raw_path, low_memory=False)
    df["approval_status"] = label
    permits_frames.append(df)

if permits_frames:
    sd_permits_raw = pd.concat(permits_frames, ignore_index=True)
    print("Optional City of SD permit appendix loaded:", sd_permits_raw.shape)
else:
    sd_permits_raw = pd.DataFrame()
    CITY_PERMITS_OPTIONAL = False
    print(
        "Skipping optional City of San Diego permit appendix. "
        "The core dashboard pipeline does not require these files."
    )

sd_permits_raw.head()


Optional City of SD permit file not found: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/raw/sandiego_permits/active_approvals_raw.csv
Optional City of SD permit file not found: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/raw/sandiego_permits/closed_approvals_raw.csv
Skipping optional City of San Diego permit appendix. The core dashboard pipeline does not require these files.


""


In [250]:
if CITY_PERMITS_OPTIONAL:
    print(sd_permits_raw.columns.tolist())
else:
    print("City of SD permit columns: skipped (optional source not loaded).")


City of SD permit columns: skipped (optional source not loaded).


In [251]:
DU_TIER_COLS = [
    "APPROVAL_DU_EXTREMELY_LOW",
    "APPROVAL_DU_VERY_LOW",
    "APPROVAL_DU_LOW",
    "APPROVAL_DU_MODERATE",
    "APPROVAL_DU_ABOVE_MODERATE",
]

ADU_JADU_COLS = [
    "APPROVAL_ADU_TOTAL",
    "APPROVAL_JADU_TOTAL",
]

CITY_PERMIT_REQUIRED_COLS = {
    "APPROVAL_ISSUE_DATE",
    "PROJECT_TITLE",
    "JOB_BC_CODE_DESCRIPTION",
    "GIS_APN",
    *DU_TIER_COLS,
    *ADU_JADU_COLS,
}

if CITY_PERMITS_OPTIONAL:
    missing_city_permit_cols = (
        CITY_PERMIT_REQUIRED_COLS - set(sd_permits_raw.columns)
    )

    if missing_city_permit_cols:
        CITY_PERMITS_OPTIONAL = False
        sd_permits_housing = pd.DataFrame()

        print(
            "Skipping optional City of San Diego permit appendix because "
            "the local CSV structure does not contain the expected fields."
        )
        print("Missing columns:", sorted(missing_city_permit_cols))

    else:
        sd_permits_raw["APPROVAL_ISSUE_DATE"] = pd.to_datetime(
            sd_permits_raw["APPROVAL_ISSUE_DATE"],
            errors="coerce",
        )

        for column in DU_TIER_COLS + ADU_JADU_COLS:
            sd_permits_raw[column] = pd.to_numeric(
                sd_permits_raw[column],
                errors="coerce",
            )

        sd_permits_raw["du_tier_total"] = (
            sd_permits_raw[DU_TIER_COLS]
            .fillna(0)
            .sum(axis=1)
        )

        sd_permits_raw["adu_jadu_total"] = (
            sd_permits_raw[ADU_JADU_COLS]
            .fillna(0)
            .sum(axis=1)
        )

        has_du_impact = (
            (sd_permits_raw["du_tier_total"] != 0)
            | (sd_permits_raw["adu_jadu_total"] != 0)
        )

        in_target_year = (
            sd_permits_raw["APPROVAL_ISSUE_DATE"].dt.year
            == TARGET_YEAR
        )

        sd_permits_housing = sd_permits_raw[
            has_du_impact & in_target_year
        ].copy()

        print(
            f"Housing-relevant City of SD permits, {TARGET_YEAR}:",
            len(sd_permits_housing),
        )
        print("Of", len(sd_permits_raw), "total raw rows")

        display(
            sd_permits_housing[
                [
                    "PROJECT_TITLE",
                    "JOB_BC_CODE_DESCRIPTION",
                    "du_tier_total",
                    "adu_jadu_total",
                ]
            ].head(10)
        )

else:
    sd_permits_housing = pd.DataFrame()
    print(
        "City of SD permit processing skipped. "
        "This is an optional validation appendix only."
    )


City of SD permit processing skipped. This is an optional validation appendix only.


## Check APR vs. City of SD permit overlap

In [252]:
apr_sd_only = sd_apr_target_year[
    sd_apr_target_year["jur_clean"] == "san diego"
]
apr_sd_units = apr_sd_only["bp_units_row"].sum()

if CITY_PERMITS_OPTIONAL and not sd_permits_housing.empty:
    city_permits_total = (
        sd_permits_housing["du_tier_total"].sum()
        + sd_permits_housing["adu_jadu_total"].sum()
    )

    print(
        f"APR, City of San Diego, {TARGET_YEAR} permitted units: "
        f"{apr_sd_units:.0f}"
    )
    print(
        f"City of SD permit system, {TARGET_YEAR} units "
        f"(DU tiers + ADU/JADU): {city_permits_total:.0f}"
    )

    if apr_sd_units:
        print(
            "Ratio (city system / APR): "
            f"{city_permits_total / apr_sd_units:.2f}"
        )
else:
    print(
        "APR vs. City of SD permit comparison skipped because "
        "the optional City permit files were not loaded."
    )


APR vs. City of SD permit comparison skipped because the optional City permit files were not loaded.


In [253]:
if CITY_PERMITS_OPTIONAL and not sd_permits_housing.empty:
    apr_sd_apns = set(
        apr_sd_only["APN"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    city_apns = set(
        sd_permits_housing["GIS_APN"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    overlap_apns = apr_sd_apns & city_apns

    print(f"APR SD APNs ({TARGET_YEAR}): {len(apr_sd_apns)}")
    print(f"City permit APNs ({TARGET_YEAR}): {len(city_apns)}")
    print(f"APNs appearing in both: {len(overlap_apns)}")

    if city_apns:
        print(
            "Share of City permit APNs also in APR: "
            f"{len(overlap_apns) / len(city_apns):.1%}"
        )
    else:
        print("No City permit APNs found.")
else:
    print(
        "APN overlap check skipped because the optional City of SD "
        "permit files were not loaded."
    )


APN overlap check skipped because the optional City of SD permit files were not loaded.


**Optional validation appendix:** when the City of San Diego permit files are
available, this section compares them with APR to confirm that they largely
describe overlapping permit activity and therefore should **not** be added
together. If the local City permit CSVs are absent or have a changed schema,
the appendix is skipped and the core RHNA/Housing Production pipeline continues.


## Missing data, unclear fields, and source limitations

In [254]:
data_quality_log = pd.DataFrame([
    {"type": "Source vintage", "source": "California DOF E-5", "item": "2025 population and housing stock estimates",
     "note": "Uses the revised 1/1/2025 estimates from the official 2026 E-5 workbook (E-5_2026_InternetVersion.xlsx), specifically the E5CityCounty2025 sheet; it does not use the 2026 estimate sheet."},
    {"type": "Missing data", "source": "APR Table A2", "item": "bp_affordable_share / co_affordable_share",
     "note": "NaN when bp_units_total or co_units_total is 0 for that jurisdiction-year (0/0), not a data error - means zero permits/completions that year, not unknown affordability"},
    {"type": "Missing data", "source": "APR Table A2", "item": "NOTES, LATITUDE/LONGITUDE, DR_TYPE, FIN_ASSIST_NAME, PRIOR_APN",
     "note": "Conditional fields - only populated for specific project types (e.g. DR_TYPE only for deed-restricted units). Sparse by design, not incomplete"},
    {"type": "Missing data", "source": "City of SD permits", "item": "ADU/JADU and income-tier DU columns",
     "note": "Blank on non-residential permit rows (electrical, plumbing, signage, etc.) - expected, filtered out of sd_permits_housing"},
    {"type": "Missing data", "source": "APR Table F (preservation)", "item": "entire table",
     "note": "Not yet loaded into this workstream. Exists as a processed file in the dashboard prototype repo (sd_apr_f_preservation_city_year.csv) but not pulled live here"},
    {"type": "Missing data", "source": "This workstream", "item": "NOAH (naturally occurring affordable housing) loss estimate",
     "note": "No published dataset exists for this - would need to be derived from ACS rent + building-age data. Not started"},
    {"type": "Missing data", "source": "HCD APR (general)", "item": "current reporting year, some jurisdictions",
     "note": "Jurisdictions can file late; a given year's data may be incomplete for months after the April 1 deadline. Check the 'not found' warning printed when loading before trusting a fresh pull"},
    {"type": "Missing data", "source": "HCD APR", "item": "units under construction",
     "note": "Not tracked anywhere in HCD's APR data - no table captures this milestone. Application, entitlement, permit, and completion stages are all available (Table A and Table A2), but under-construction status is not"},
    {"type": "Missing data", "source": "APR Table A2, City of San Diego", "item": "ent_units_total near-zero relative to bp_units_total",
     "note": "San Diego reported only 3 entitled units in 2025 against 7,842 permitted units - an unusually large gap for the county's largest jurisdiction. Likely a self-reporting gap in San Diego's own APR submission for the entitlement stage specifically, not evidence that entitlement activity actually stopped"},
    {"type": "Unclear field", "source": "APR Table A2", "item": "*_DR / *_NDR column suffixes",
     "note": "Deed Restricted / Non-Deed Restricted (whether the affordability requirement is legally recorded on the property). Not explained in the CSV itself - confirmed from HCD's separate Table A2 data dictionary (.docx)"},
    {"type": "Unclear field", "source": "APR Table A2 vs. RHNA6", "item": "VLOW vs. VLI",
     "note": "Same income tier (Very Low Income), different abbreviation between two HCD datasets. Easy to miss if joining/comparing by tier name"},
    {"type": "Unclear field", "source": "RHNA6 progress report", "item": "what \"units reported\" actually measures",
     "note": "RHNA progress in this dataset is based on BUILDING PERMITS ISSUED, not completed units - confirmed via HCD's own APR guidance (\"only building permits are used for the purposes of determining progress towards RHNA\"). Entitlements and completions are tracked elsewhere in the APR but do not count toward RHNA credit"},
    {"type": "Unclear field", "source": "City of SD permits", "item": "APPROVAL_DU_NET_CHANGE",
     "note": "Misleadingly named - sums to exactly 0 across a full year of data, unreliable. Real unit counts live in the separate income-tier APPROVAL_DU_* columns instead"},
    {"type": "Resolved / corrected", "source": "APR Table A2", "item": "NO_ENTITLEMENTS / NO_BUILDING_PERMITS / NO_OTHER_FORMS_OF_READINESS",
     "note": "CORRECTED: previously mischaracterized as flags. Per HCD's official APR instructions, these mean \"Number Of\" - auto-populated total-unit-count fields calculated by HCD from the same per-tier income columns this notebook sums independently. Cross-check against ent_units_total / bp_units_total / co_units_total now matches exactly for all 19/19 jurisdictions, all three stages"},
    {"type": "Resolved / corrected", "source": "This notebook", "item": "ENT_INCOME_COLS incorrectly included EXTR_LOW_INCOME_UNITS",
     "note": "A loose text search (\"contains INCOME, not BP_/CO_ prefixed\") swept in an unrelated Table A2 field, inflating ent_units_total beyond what the 4 tier columns summed to. Caused an apparent 55-unit Escondido anomaly that was never a real data-source issue - fixed by building the column list explicitly from the known tier structure instead of a text search"},
    {"type": "Resolved / documented", "source": "APR Table A", "item": "TOT_PROPOSED_UNITS vs. income-tier application totals",
     "note": "Overall application totals use HCD TOT_PROPOSED_UNITS and are exported as income_category='all'. HCD income-tier fields do not always reconcile exactly to that total and can sum either above or below it, so Power BI must not reconstruct the overall application count by summing tier rows. application_unclassified_total retains only a positive overall-minus-tier gap."},
    {"type": "Missing data", "source": "APR Table A (historical, 2018-2020)", "item": "jurisdiction-year application records absent from source",
     "note": "Lemon Grove (2018), Imperial Beach (2019), and San Marcos (2020) are absent from the raw HCD Table A records. They remain NaN and are labeled 'missing from source' instead of being converted to zero."},
    {"type": "Unclear field", "source": "HCD / DOF / dashboard prototype", "item": "County jurisdiction naming",
     "note": "Appears as 'Unincorporated' (DOF), 'SAN DIEGO COUNTY' (APR/RHNA), and 'County of San Diego' in various places - required manual normalization to a single consistent key ('unincorporated san diego county') across this notebook"},
    {"type": "Source limitation", "source": "APR / RHNA (HCD)", "item": "self-reported",
     "note": "Not independently verified by HCD. Quality, completeness, and filing timeliness vary by jurisdiction"},
    {"type": "Source limitation", "source": "RHNA6 progress report", "item": "cumulative only",
     "note": "No year-by-year breakdown - reports cycle-to-date totals only (2021-2029), can't see year-over-year pace toward the target from this file alone"},
    {"type": "Source limitation", "source": "City of SD permits", "item": "single-jurisdiction coverage",
     "note": "Only covers the City of San Diego, not the other 17 jurisdictions. Also overlaps with APR for San Diego specifically (~1.04 ratio, 93% APN match) - don't sum the two for San Diego totals"},
    {"type": "Source limitation", "source": "DOF E-5", "item": "annual point-in-time estimate",
     "note": "January 1 snapshot, not real-time. Uses different methodology than ACS, so the two won't match exactly even for the same year"},
    {"type": "Source limitation", "source": "Census ACS", "item": "one year behind target year",
     "note": "Newest available vintage is 2020-2024 (\"2024\" data) while APR/DOF/permits target 2025 - ACS structurally cannot produce 2025 data until ~Dec 2026/Jan 2027. Also a 5-year rolling estimate with margins of error, largest for small jurisdictions like Del Mar (~3,900 population)"},
    {"type": "Source limitation", "source": "Dashboard prototype (housing-dashboard-prototype repo)", "item": "stale baseline for validation",
     "note": "The prototype's sd_apr_a2_city_year_supply.csv only covers 2018-2024 - it has no 2025 data yet, so this notebook's 2025 pull currently has nothing to validate against for that year. Not an error in this notebook; the prototype simply hasn't been refreshed with 2025 APR data"},
    {"type": "Source limitation", "source": "HCD APR (general)", "item": "historical years get amended after publication",
     "note": "Validating 2018-2024 against the dashboard prototype found 9 of 126 city-year rows differ (out of 18 cities x 7 years). In every case this notebook's fresh pull is HIGHER than the prototype's older snapshot, never lower, and gaps are small (3-59 units) - consistent with HCD amending past years' data (late filings, corrections) after initial publication. Largest gap: San Marcos 2024 (455 vs. 396, ~13%)"},
])

data_quality_log.to_csv(DOCS_DIR / "data_quality_limitations.csv", index=False)
data_quality_log


,type,source,item,note
0,Source vintage,California DOF E-5,2025 population and housing stock estimates,Uses the revised 1/1/2025 estimates from the o...
1,Missing data,APR Table A2,bp_affordable_share / co_affordable_share,NaN when bp_units_total or co_units_total is 0...
2,Missing data,APR Table A2,"NOTES, LATITUDE/LONGITUDE, DR_TYPE, FIN_ASSIST...",Conditional fields - only populated for specif...
3,Missing data,City of SD permits,ADU/JADU and income-tier DU columns,Blank on non-residential permit rows (electric...
4,Missing data,APR Table F (preservation),entire table,Not yet loaded into this workstream. Exists as...
5,Missing data,This workstream,NOAH (naturally occurring affordable housing) ...,No published dataset exists for this - would n...
6,Missing data,HCD APR (general),"current reporting year, some jurisdictions",Jurisdictions can file late; a given year's da...
7,Missing data,HCD APR,units under construction,Not tracked anywhere in HCD's APR data - no ta...
8,Missing data,"APR Table A2, City of San Diego",ent_units_total near-zero relative to bp_units...,San Diego reported only 3 entitled units in 20...
9,Unclear field,APR Table A2,*_DR / *_NDR column suffixes,Deed Restricted / Non-Deed Restricted (whether...


# Final Workstream 1 additions

The following cells complete the remaining Housing Production requirements:

1. Annual permitted and completed units by HCD housing type;
2. Housing-stock benchmark years (2000, 2010, 2021, 2024);
3. Annual DOF E-5 housing-stock estimates for 2020-2025;
4. Power BI-ready exports, validation, documentation, and a final completion checklist.

HCD's source categories combine **2-, 3-, and 4-unit buildings** into one
`2-4` category. The notebook does not invent a more detailed duplex versus
3-4-unit split when the source does not provide it.


In [255]:
# ============================================================
# Housing Production by Housing Type, 2018-2025
# ============================================================

HCD_HOUSING_TYPE_LABELS = {
    "SFD": "Single-family detached",
    "SFA": "Single-family attached",
    "2-4": "2-4 unit building (combined)",
    "2 TO 4": "2-4 unit building (combined)",
    "2 TO 4 UNITS": "2-4 unit building (combined)",
    "2 - 4": "2-4 unit building (combined)",
    "5+": "5+ unit building",
    "5 +": "5+ unit building",
    "ADU": "Accessory dwelling unit",
    "MH": "Mobile home / manufactured home",
}

HCD_STANDARD_HOUSING_TYPES = [
    "Single-family detached",
    "Single-family attached",
    "2-4 unit building (combined)",
    "5+ unit building",
    "Accessory dwelling unit",
    "Mobile home / manufactured home",
    "Other / source-reported",
    "Unknown / not reported",
]


def normalize_hcd_housing_type(value):
    if pd.isna(value):
        return "Unknown / not reported"

    text = re.sub(
        r"\s+",
        " ",
        str(value).strip().upper(),
    )

    if not text:
        return "Unknown / not reported"

    return HCD_HOUSING_TYPE_LABELS.get(
        text,
        "Other / source-reported",
    )


apr_type_base = sd_apr[
    sd_apr[YEAR_COL].isin(APR_YEARS)
].copy()

apr_type_base["housing_type"] = (
    apr_type_base["UNIT_CAT"]
    .map(normalize_hcd_housing_type)
)

production_type_parts = []

for stage_label, value_column in {
    "Building Permit": "bp_units_row",
    "Completion": "co_units_row",
}.items():

    observed = (
        apr_type_base
        .groupby(
            ["jur_clean", YEAR_COL, "housing_type"],
            as_index=False,
        )[value_column]
        .sum()
        .rename(
            columns={
                YEAR_COL: "year",
                value_column: "value",
            }
        )
    )

    grid = pd.MultiIndex.from_product(
        [
            sorted(sd_jur_keys),
            APR_YEARS,
            HCD_STANDARD_HOUSING_TYPES,
        ],
        names=[
            "jur_clean",
            "year",
            "housing_type",
        ],
    ).to_frame(index=False)

    grid = grid.merge(
        observed,
        on=[
            "jur_clean",
            "year",
            "housing_type",
        ],
        how="left",
    )

    coverage = production_history[
        [
            "jur_clean",
            "year",
            "reporting_status",
        ]
    ].copy()

    grid = grid.merge(
        coverage,
        on=["jur_clean", "year"],
        how="left",
    )

    # A missing type within a jurisdiction-year that did report APR is a true
    # zero for that category. A missing jurisdiction-year remains NaN.
    grid["value"] = np.where(
        grid["reporting_status"].eq("reported"),
        grid["value"].fillna(0),
        np.nan,
    )

    grid["development_stage"] = stage_label
    grid["jurisdiction"] = grid["jur_clean"].map(
        JURISDICTION_DISPLAY
    )
    grid["geographic_level"] = "Jurisdiction"
    grid["source"] = "HCD APR Table A2"
    grid["source_year"] = grid["year"]
    grid["download_date"] = DOWNLOAD_DATE
    grid["unit_of_measurement"] = "Housing units (count)"
    grid["limitations"] = (
        "HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and MH. "
        "The 2-4 category cannot be defensibly split into duplex versus "
        "3-4-unit buildings from this source alone. Stage activity is counted "
        "only when the stage date is in the APR reporting year."
    )

    production_type_parts.append(grid)

jurisdiction_production_by_type = pd.concat(
    production_type_parts,
    ignore_index=True,
)

# Regional totals. A regional value is complete only if all 19
# jurisdiction rows for that year/type/stage are present.
regional_production_by_type = (
    jurisdiction_production_by_type
    .groupby(
        [
            "year",
            "development_stage",
            "housing_type",
            "source",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        value=(
            "value",
            lambda s: (
                s.sum()
                if s.notna().sum() == 19
                else np.nan
            ),
        ),
        jurisdictions_reporting=("value", "count"),
    )
)

regional_production_by_type["jurisdiction"] = REGION_NAME
regional_production_by_type["geographic_level"] = "Regional total"
regional_production_by_type["reporting_status"] = np.where(
    regional_production_by_type["jurisdictions_reporting"].eq(19),
    "complete",
    "partial",
)
regional_production_by_type["source_year"] = (
    regional_production_by_type["year"]
)
regional_production_by_type["download_date"] = DOWNLOAD_DATE
regional_production_by_type["unit_of_measurement"] = (
    "Housing units (count)"
)
regional_production_by_type["limitations"] = (
    "Regional sum of all 18 cities plus Unincorporated San Diego County. "
    "If any jurisdiction-year is missing from APR, the regional type value "
    "is left blank rather than presented as a complete total."
)

POWERBI_TYPE_COLUMNS = [
    "jurisdiction",
    "year",
    "development_stage",
    "housing_type",
    "value",
    "reporting_status",
    "source",
    "source_year",
    "download_date",
    "geographic_level",
    "unit_of_measurement",
    "limitations",
]

powerbi_production_by_type = pd.concat(
    [
        jurisdiction_production_by_type[
            POWERBI_TYPE_COLUMNS
        ],
        regional_production_by_type[
            POWERBI_TYPE_COLUMNS
        ],
    ],
    ignore_index=True,
)

powerbi_production_by_type = (
    powerbi_production_by_type
    .sort_values(
        [
            "jurisdiction",
            "year",
            "development_stage",
            "housing_type",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validation: type rows must reconcile to total permits/completions.
# ------------------------------------------------------------

for stage_label, expected_column in {
    "Building Permit": "bp_units_total",
    "Completion": "co_units_total",
}.items():

    typed_total = (
        jurisdiction_production_by_type[
            jurisdiction_production_by_type[
                "development_stage"
            ].eq(stage_label)
        ]
        .groupby(
            ["jur_clean", "year"]
        )["value"]
        .sum(min_count=1)
    )

    expected_total = (
        production_history
        .set_index(
            ["jur_clean", "year"]
        )[expected_column]
    )

    assert np.allclose(
        typed_total.reindex(
            expected_total.index
        ),
        expected_total,
        equal_nan=True,
    ), (
        f"Housing-type {stage_label} totals do not reconcile "
        f"to {expected_column}."
    )

production_type_path = (
    PROCESSED_DIR
    / f"powerbi_production_by_housing_type_{APR_START_YEAR}_{TARGET_YEAR}.csv"
)

powerbi_production_by_type.to_csv(
    production_type_path,
    index=False,
)

housing_type_mapping = pd.DataFrame(
    [
        {
            "source_code": code,
            "dashboard_label": label,
        }
        for code, label in HCD_HOUSING_TYPE_LABELS.items()
    ]
).drop_duplicates()

housing_type_mapping.to_csv(
    DOCS_DIR / "hcd_housing_type_mapping.csv",
    index=False,
)

print("Housing-type production validation passed.")
print("Saved:", production_type_path)
print("Rows:", len(powerbi_production_by_type))
powerbi_production_by_type.head()


Housing-type production validation passed.
Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/powerbi_production_by_housing_type_2018_2025.csv
Rows: 2560


,jurisdiction,year,development_stage,housing_type,value,reporting_status,source,source_year,download_date,geographic_level,unit_of_measurement,limitations
0,Carlsbad,2018,Building Permit,2-4 unit building (combined),11.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
1,Carlsbad,2018,Building Permit,5+ unit building,49.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
2,Carlsbad,2018,Building Permit,Accessory dwelling unit,33.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
3,Carlsbad,2018,Building Permit,Mobile home / manufactured home,0.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."
4,Carlsbad,2018,Building Permit,Other / source-reported,0.0,reported,HCD APR Table A2,2018,2026-08-17,Jurisdiction,Housing units (count),"HCD UNIT_CAT uses SFD, SFA, 2-4, 5+, ADU, and ..."


## Housing stock benchmark years

The benchmark table uses the same total-housing-unit concept across the
requested benchmark years wherever possible:

- 2000 Decennial Census SF1: `H001001`
- 2010 Decennial Census SF1: `H001001`
- 2021 ACS 5-year: `B25001_001E`
- 2024 ACS 5-year: `B25001_001E`

The countywide value is pulled directly. The unincorporated value is a
transparent residual (county total minus the 18 incorporated cities) because
these place/county benchmark APIs do not provide a directly comparable
unincorporated-jurisdiction row.


In [256]:
# ============================================================
# Housing Stock Benchmark Years
# 2000 Census, 2010 Census, 2021 ACS 5-year, 2024 ACS 5-year
# ============================================================

CENSUS_BENCHMARKS = [
    {
        "year": 2000,
        "label": "2000 Census",
        "dataset": "2000 Decennial Census SF1",
        "api_url": "https://api.census.gov/data/2000/dec/sf1",
        "variable": "H001001",
        "estimate_type": "Decennial Census count",
    },
    {
        "year": 2010,
        "label": "2010 Census",
        "dataset": "2010 Decennial Census SF1",
        "api_url": "https://api.census.gov/data/2010/dec/sf1",
        "variable": "H001001",
        "estimate_type": "Decennial Census count",
    },
    {
        "year": 2021,
        "label": "2021 ACS 5-year",
        "dataset": "2021 ACS 5-year",
        "api_url": "https://api.census.gov/data/2021/acs/acs5",
        "variable": "B25001_001E",
        "estimate_type": "ACS 5-year estimate",
    },
    {
        "year": 2024,
        "label": "2024 ACS 5-year",
        "dataset": "2024 ACS 5-year",
        "api_url": "https://api.census.gov/data/2024/acs/acs5",
        "variable": "B25001_001E",
        "estimate_type": "ACS 5-year estimate",
    },
]


def normalize_census_place_name(name):
    text = str(name)

    text = re.sub(
        r",\s*California$",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s+(city|town|village|CDP)$",
        "",
        text,
        flags=re.IGNORECASE,
    )

    return normalize_jurisdiction(text)


def census_housing_benchmark(config):
    variable = config["variable"]
    api_url = config["api_url"]

    place_params = {
        "get": f"NAME,{variable}",
        "for": "place:*",
        "in": f"state:{CALIFORNIA_STATE_FIPS}",
        "key": CENSUS_API_KEY,
    }

    county_params = {
        "get": f"NAME,{variable}",
        "for": f"county:{SAN_DIEGO_COUNTY_FIPS}",
        "in": f"state:{CALIFORNIA_STATE_FIPS}",
        "key": CENSUS_API_KEY,
    }

    place_raw = get_json(
        api_url,
        params=place_params,
    )

    county_raw = get_json(
        api_url,
        params=county_params,
    )

    places = pd.DataFrame(
        place_raw[1:],
        columns=place_raw[0],
    )

    county = pd.DataFrame(
        county_raw[1:],
        columns=county_raw[0],
    )

    places["jur_clean"] = (
        places["NAME"]
        .map(normalize_census_place_name)
    )

    places[variable] = pd.to_numeric(
        places[variable],
        errors="coerce",
    )

    county[variable] = pd.to_numeric(
        county[variable],
        errors="coerce",
    )

    sd_cities = places[
        places["jur_clean"].isin(
            {
                normalize_jurisdiction(city)
                for city in SAN_DIEGO_CITIES
            }
        )
    ].copy()

    assert sd_cities["jur_clean"].nunique() == 18, (
        f"{config['label']}: expected 18 incorporated San Diego "
        f"cities, found {sd_cities['jur_clean'].nunique()}."
    )

    county_total = float(
        county[variable].iloc[0]
    )

    incorporated_total = float(
        sd_cities[variable].sum()
    )

    unincorporated_residual = (
        county_total - incorporated_total
    )

    assert unincorporated_residual >= 0, (
        f"{config['label']}: derived unincorporated housing units "
        "were negative."
    )

    records = []

    for _, row in sd_cities.iterrows():
        jur_clean = row["jur_clean"]

        records.append(
            {
                "jurisdiction": JURISDICTION_DISPLAY[jur_clean],
                "jur_clean": jur_clean,
                "benchmark_year": config["year"],
                "benchmark_label": config["label"],
                "housing_units_total": float(row[variable]),
                "source": "U.S. Census Bureau",
                "source_dataset": config["dataset"],
                "source_variable": variable,
                "source_url": api_url,
                "geographic_level": "Jurisdiction",
                "estimate_type": config["estimate_type"],
                "derivation": "Direct published place value",
                "download_date": DOWNLOAD_DATE,
                "limitations": (
                    "Uses the Census/ACS total housing-unit definition. "
                    "ACS values are estimates and differ methodologically "
                    "from Decennial Census counts and DOF E-5 estimates."
                ),
            }
        )

    # Census APIs do not publish an incorporated-place-equivalent
    # "Unincorporated San Diego County" row in these benchmark tables.
    # Derive it transparently as County total minus the 18 incorporated cities.
    records.append(
        {
            "jurisdiction": "Unincorporated San Diego County",
            "jur_clean": "unincorporated san diego county",
            "benchmark_year": config["year"],
            "benchmark_label": config["label"],
            "housing_units_total": unincorporated_residual,
            "source": "U.S. Census Bureau",
            "source_dataset": config["dataset"],
            "source_variable": variable,
            "source_url": api_url,
            "geographic_level": "Jurisdiction",
            "estimate_type": config["estimate_type"],
            "derivation": (
                "Derived residual: San Diego County total minus "
                "the 18 incorporated city totals"
            ),
            "download_date": DOWNLOAD_DATE,
            "limitations": (
                "Derived residual rather than a directly published "
                "unincorporated benchmark. For ACS years, this combines "
                "survey estimates; margins of error are not propagated "
                "in this dashboard value."
            ),
        }
    )

    records.append(
        {
            "jurisdiction": "San Diego County (Countywide)",
            "jur_clean": "san diego county countywide",
            "benchmark_year": config["year"],
            "benchmark_label": config["label"],
            "housing_units_total": county_total,
            "source": "U.S. Census Bureau",
            "source_dataset": config["dataset"],
            "source_variable": variable,
            "source_url": api_url,
            "geographic_level": "Countywide",
            "estimate_type": config["estimate_type"],
            "derivation": "Direct published county value",
            "download_date": DOWNLOAD_DATE,
            "limitations": (
                "Countywide value includes incorporated and "
                "unincorporated areas."
            ),
        }
    )

    return pd.DataFrame(records)


benchmark_parts = [
    census_housing_benchmark(config)
    for config in CENSUS_BENCHMARKS
]

housing_stock_benchmarks = pd.concat(
    benchmark_parts,
    ignore_index=True,
)

assert (
    housing_stock_benchmarks
    .groupby("benchmark_year")
    .size()
    .eq(20)
    .all()
), "Expected 20 geographic rows for every benchmark year."

assert not housing_stock_benchmarks.duplicated(
    subset=[
        "jurisdiction",
        "benchmark_year",
    ]
).any(), "Duplicate housing-stock benchmark keys found."

assert (
    housing_stock_benchmarks[
        "housing_units_total"
    ] >= 0
).all(), "Negative housing-stock benchmark found."

benchmark_path = (
    PROCESSED_DIR
    / "powerbi_housing_stock_benchmarks_2000_2010_2021_2024.csv"
)

housing_stock_benchmarks.to_csv(
    benchmark_path,
    index=False,
)

print("Housing-stock benchmark validation passed.")
print("Saved:", benchmark_path)
print(housing_stock_benchmarks.shape)
housing_stock_benchmarks.head()


Housing-stock benchmark validation passed.
Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/powerbi_housing_stock_benchmarks_2000_2010_2021_2024.csv
(80, 14)


,jurisdiction,jur_clean,benchmark_year,benchmark_label,housing_units_total,source,source_dataset,source_variable,source_url,geographic_level,estimate_type,derivation,download_date,limitations
0,Carlsbad,carlsbad,2000,2000 Census,33798.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
1,Chula Vista,chula vista,2000,2000 Census,59495.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
2,Coronado,coronado,2000,2000 Census,9494.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
3,Del Mar,del mar,2000,2000 Census,2557.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...
4,El Cajon,el cajon,2000,2000 Census,35190.0,U.S. Census Bureau,2000 Decennial Census SF1,H001001,https://api.census.gov/data/2000/dec/sf1,Jurisdiction,Decennial Census count,Direct published place value,2026-08-17,Uses the Census/ACS total housing-unit definit...


## Annual DOF E-5 housing stock

This table uses the annual `E5CityCountyYYYY` sheets from the official DOF
2026 workbook but stops at the dashboard target year (2025). These values are
annual January 1 housing-stock estimates, not annual production.


In [257]:
# ============================================================
# Annual DOF E-5 Housing Stock, 2020-2025
# Uses annual sheets from the official 2026 release workbook.
# ============================================================

DOF_ANNUAL_YEARS = list(
    range(2020, TARGET_YEAR + 1)
)

DOF_ANNUAL_METRICS = {
    "total_housing_units": "housing_units_total",
    "occupied_housing_units": "Occupied",
    "vacant_housing_units": "vacant_units",
    "single_family_detached": "Single Detached",
    "single_family_attached": "Single Attached",
    "single_family_total": "single_family_units",
    "two_to_four_units": "Two to Four",
    "five_plus_units": "Five Plus",
    "multifamily_total": "multifamily_units",
    "mobile_homes": "Mobile Homes",
}


def load_dof_year(year):
    sheet_name = f"E5CityCounty{year}"

    raw = pd.read_excel(
        DOF_RAW_PATH,
        sheet_name=sheet_name,
        header=3,
    )

    raw = raw.rename(
        columns={
            "County/City/State": "name",
            "Total": "population_total",
            "Total.1": "housing_units_total",
        }
    )

    raw["name"] = (
        raw["name"]
        .astype(str)
        .str.strip()
    )

    raw["is_county_header"] = (
        raw["population_total"].isna()
        & raw["name"].str.contains(
            "County",
            na=False,
        )
    )

    raw["county"] = (
        raw["name"]
        .where(raw["is_county_header"])
        .ffill()
    )

    sd_block = raw[
        raw["county"].eq("San Diego County")
    ].copy()

    jurisdictions = sd_block[
        (~sd_block["is_county_header"])
        & sd_block["population_total"].notna()
        & (~sd_block["name"].isin(
            [
                "Incorporated",
                "County Total",
            ]
        ))
    ].copy()

    jurisdictions["jur_clean"] = (
        jurisdictions["name"]
        .map(
            lambda n: (
                normalize_jurisdiction(
                    COUNTY_JURISDICTION_NAME
                )
                if n == "Unincorporated"
                else normalize_jurisdiction(n)
            )
        )
    )

    assert len(jurisdictions) == 19, (
        f"DOF {year}: expected 19 jurisdiction rows, "
        f"got {len(jurisdictions)}."
    )

    county_total = sd_block[
        sd_block["name"].eq("County Total")
    ].copy()

    assert len(county_total) == 1, (
        f"DOF {year}: expected exactly one County Total row."
    )

    combined = jurisdictions.copy()

    combined["jurisdiction"] = (
        combined["jur_clean"]
        .map(JURISDICTION_DISPLAY)
    )

    combined["geographic_level"] = "Jurisdiction"

    county_total["jur_clean"] = (
        "san diego county countywide"
    )

    county_total["jurisdiction"] = (
        "San Diego County (Countywide)"
    )

    county_total["geographic_level"] = "Countywide"

    combined = pd.concat(
        [
            combined,
            county_total,
        ],
        ignore_index=True,
        sort=False,
    )

    numeric_source_columns = [
        "housing_units_total",
        "Single Detached",
        "Single Attached",
        "Two to Four",
        "Five Plus",
        "Mobile Homes",
        "Occupied",
    ]

    for column in numeric_source_columns:
        combined[column] = pd.to_numeric(
            combined[column],
            errors="coerce",
        )

    combined["vacant_units"] = (
        combined["housing_units_total"]
        - combined["Occupied"]
    )

    combined["single_family_units"] = (
        combined["Single Detached"]
        + combined["Single Attached"]
    )

    combined["multifamily_units"] = (
        combined["Two to Four"]
        + combined["Five Plus"]
    )

    structure_sum = (
        combined["Single Detached"]
        + combined["Single Attached"]
        + combined["Two to Four"]
        + combined["Five Plus"]
        + combined["Mobile Homes"]
    )

    assert np.allclose(
        structure_sum,
        combined["housing_units_total"],
        equal_nan=True,
    ), (
        f"DOF {year}: housing-type categories do not "
        "reconcile to total housing units."
    )

    combined["year"] = year

    return combined


dof_annual_wide = pd.concat(
    [
        load_dof_year(year)
        for year in DOF_ANNUAL_YEARS
    ],
    ignore_index=True,
)

assert (
    dof_annual_wide
    .groupby("year")
    .size()
    .eq(20)
    .all()
), "Expected 20 DOF geographic rows for every year."

dof_long_parts = []

for metric_label, source_column in DOF_ANNUAL_METRICS.items():
    chunk = dof_annual_wide[
        [
            "jurisdiction",
            "jur_clean",
            "year",
            "geographic_level",
            source_column,
        ]
    ].copy()

    chunk = chunk.rename(
        columns={
            source_column: "value",
        }
    )

    chunk["metric"] = metric_label
    chunk["source"] = (
        "California Department of Finance E-5"
    )
    chunk["source_release"] = (
        "2026 E-5 release"
    )
    chunk["source_year"] = chunk["year"]
    chunk["download_date"] = DOWNLOAD_DATE
    chunk["unit_of_measurement"] = (
        "Housing units (count)"
    )
    chunk["limitations"] = (
        "January 1 annual estimate from the official 2026 E-5 "
        "workbook. Historical sheets in that release may contain "
        "revised estimates. This is housing stock, not annual production."
    )

    dof_long_parts.append(chunk)

powerbi_dof_annual_stock = pd.concat(
    dof_long_parts,
    ignore_index=True,
)

assert not powerbi_dof_annual_stock.duplicated(
    subset=[
        "jurisdiction",
        "year",
        "metric",
    ]
).any(), "Duplicate DOF annual housing-stock keys found."

dof_annual_path = (
    PROCESSED_DIR
    / f"powerbi_dof_annual_housing_stock_2020_{TARGET_YEAR}.csv"
)

powerbi_dof_annual_stock.to_csv(
    dof_annual_path,
    index=False,
)

print("DOF annual housing-stock validation passed.")
print("Saved:", dof_annual_path)
print(powerbi_dof_annual_stock.shape)
powerbi_dof_annual_stock.head()


DOF annual housing-stock validation passed.
Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/powerbi_dof_annual_housing_stock_2020_2025.csv
(1200, 12)


,jurisdiction,jur_clean,year,geographic_level,value,metric,source,source_release,source_year,download_date,unit_of_measurement,limitations
0,Carlsbad,carlsbad,2020,Jurisdiction,47734.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
1,Chula Vista,chula vista,2020,Jurisdiction,87284.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
2,Coronado,coronado,2020,Jurisdiction,9573.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
3,Del Mar,del mar,2020,Jurisdiction,2574.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...
4,El Cajon,el cajon,2020,Jurisdiction,36749.0,total_housing_units,California Department of Finance E-5,2026 E-5 release,2020,2026-08-17,Housing units (count),January 1 annual estimate from the official 20...


In [258]:
# ============================================================
# Final Workstream 1 documentation + Power BI manifest
# ============================================================

additional_metric_dictionary = pd.DataFrame(
    [
        {
            "output_metric": "production_by_housing_type",
            "source_table": "HCD APR Table A2",
            "source_variables": "UNIT_CAT + BP_* / CO_* unit fields + stage dates",
            "definition": (
                "Annual building-permit and completion units by HCD housing "
                "type and jurisdiction."
            ),
            "source_year": f"{APR_START_YEAR}-{TARGET_YEAR}",
            "download_date": DOWNLOAD_DATE,
            "geographic_level": "Jurisdiction and regional total",
            "unit_of_measurement": "Housing units (count)",
            "limitations": (
                "HCD combines duplex, triplex, and fourplex units in one "
                "2-4 category. The dashboard does not fabricate a finer split."
            ),
        },
        {
            "output_metric": "housing_stock_benchmark_total",
            "source_table": "U.S. Census Bureau Decennial Census / ACS",
            "source_variables": "H001001 (2000/2010); B25001_001E (2021/2024)",
            "definition": (
                "Total housing units at benchmark years 2000, 2010, 2021, "
                "and 2024."
            ),
            "source_year": "2000, 2010, 2021, 2024",
            "download_date": DOWNLOAD_DATE,
            "geographic_level": (
                "18 cities, derived unincorporated county, and countywide"
            ),
            "unit_of_measurement": "Housing units (count)",
            "limitations": (
                "ACS values are survey estimates; unincorporated benchmark "
                "values are derived as county total minus the 18 city totals."
            ),
        },
        {
            "output_metric": "dof_annual_housing_stock",
            "source_table": "California DOF E-5",
            "source_variables": (
                "Total housing units, Occupied, Single Detached, "
                "Single Attached, Two to Four, Five Plus, Mobile Homes"
            ),
            "definition": (
                "Annual January 1 housing-stock estimates from 2020 through "
                f"{TARGET_YEAR}, including occupancy and structure categories."
            ),
            "source_year": f"2020-{TARGET_YEAR}",
            "download_date": DOWNLOAD_DATE,
            "geographic_level": (
                "18 cities, Unincorporated San Diego County, and countywide"
            ),
            "unit_of_measurement": "Housing units (count)",
            "limitations": (
                "Point-in-time DOF estimate; not annual production. Uses the "
                "annual sheets in the official 2026 E-5 workbook."
            ),
        },
    ]
)

metric_dictionary = pd.concat(
    [
        metric_dictionary,
        additional_metric_dictionary,
    ],
    ignore_index=True,
)

metric_dictionary = metric_dictionary.drop_duplicates(
    subset=["output_metric"],
    keep="last",
)

metric_dictionary.to_csv(
    DOCS_DIR
    / "rhna_housing_production_metric_dictionary.csv",
    index=False,
)


additional_export_metadata = pd.DataFrame(
    [
        {
            "file": production_type_path.name,
            "description": (
                "Annual building-permit and completion units by housing type "
                "for Power BI."
            ),
            "dashboard_tab": "Housing Production",
            "grain": (
                "jurisdiction / year / development stage / housing type"
            ),
        },
        {
            "file": benchmark_path.name,
            "description": (
                "Housing-stock benchmark totals for 2000, 2010, 2021, 2024."
            ),
            "dashboard_tab": "Housing Production",
            "grain": "jurisdiction / benchmark year",
        },
        {
            "file": dof_annual_path.name,
            "description": (
                "Annual DOF E-5 housing-stock estimates, occupancy, and "
                "structure categories."
            ),
            "dashboard_tab": "Housing Production",
            "grain": "jurisdiction / year / stock metric",
        },
    ]
)

# Keep existing export metadata if it already exists, then append only
# the three new Power BI tables.
if "export_metadata" in globals():
    export_metadata_complete = pd.concat(
        [
            export_metadata,
            additional_export_metadata,
        ],
        ignore_index=True,
        sort=False,
    )
else:
    export_metadata_complete = additional_export_metadata.copy()

export_metadata_complete.to_csv(
    DOCS_DIR / "export_metadata.csv",
    index=False,
)


additional_quality_notes = pd.DataFrame(
    [
        {
            "type": "Methodology",
            "source": "HCD APR Table A2",
            "item": "Annual stage-date filtering",
            "note": (
                "Entitlement, building-permit, and completion units are counted "
                "for an APR year only when the corresponding stage date falls "
                "within that reporting year. This prevents prior-stage values "
                "carried on a project row from being treated as new annual activity."
            ),
        },
        {
            "type": "Source limitation",
            "source": "HCD APR Table A2",
            "item": "Housing type detail",
            "note": (
                "HCD's UNIT_CAT provides SFD, SFA, 2-4, 5+, ADU, and MH. "
                "The 2-4 category cannot be split into duplex versus 3-4-unit "
                "buildings without another source, so the dashboard keeps it combined."
            ),
        },
        {
            "type": "Derived geography",
            "source": "U.S. Census Bureau",
            "item": "Unincorporated benchmark housing stock",
            "note": (
                "For 2000, 2010, 2021, and 2024 benchmark tables, the "
                "unincorporated value is derived as the published San Diego "
                "County total minus the 18 incorporated city totals. ACS margins "
                "of error are not propagated for this residual."
            ),
        },
        {
            "type": "Source limitation",
            "source": "HCD APR",
            "item": "Units under construction",
            "note": (
                "A consistent regionwide under-construction measure is not "
                "available in the APR fields used here. The dashboard should "
                "show this stage only if a separate defensible source is added."
            ),
        },
        {
            "type": "Methodology",
            "source": "Workstream 1 requirements",
            "item": "Housing deficit",
            "note": (
                "No standalone housing-deficit metric is calculated because "
                "the requirements specify that a deficit should be shown only "
                "when a clearly defined and defensible need measure is available. "
                "Existing stock, RHNA allocation, and remaining RHNA need remain "
                "separate measures."
            ),
        },
    ]
)

if "data_quality_log" in globals():
    data_quality_log = pd.concat(
        [
            data_quality_log,
            additional_quality_notes,
        ],
        ignore_index=True,
    )
else:
    data_quality_log = additional_quality_notes.copy()

data_quality_log.to_csv(
    DOCS_DIR / "data_quality_limitations.csv",
    index=False,
)


workstream1_powerbi_manifest = pd.DataFrame(
    [
        {
            "dashboard_tab": "RHNA",
            "file": (
                f"powerbi_rhna_production_{APR_START_YEAR}_{TARGET_YEAR}_long.csv"
            ),
            "use_for": (
                "RHNA allocation/progress/remaining/% complete plus separate "
                "applications, entitlements, permits, and completions."
            ),
        },
        {
            "dashboard_tab": "Housing Production",
            "file": production_type_path.name,
            "use_for": (
                "Permitted and completed housing units by year, jurisdiction, "
                "and HCD housing type."
            ),
        },
        {
            "dashboard_tab": "Housing Production",
            "file": benchmark_path.name,
            "use_for": (
                "2000/2010/2021/2024 total-housing-stock benchmark comparison."
            ),
        },
        {
            "dashboard_tab": "Housing Production",
            "file": dof_annual_path.name,
            "use_for": (
                "Annual 2020-2025 housing-stock totals, occupied/vacant units, "
                "and DOF structure categories."
            ),
        },
    ]
)

workstream1_powerbi_manifest.to_csv(
    DOCS_DIR / "workstream1_powerbi_manifest.csv",
    index=False,
)

display(workstream1_powerbi_manifest)


,dashboard_tab,file,use_for
0,RHNA,powerbi_rhna_production_2018_2025_long.csv,RHNA allocation/progress/remaining/% complete ...
1,Housing Production,powerbi_production_by_housing_type_2018_2025.csv,"Permitted and completed housing units by year,..."
2,Housing Production,powerbi_housing_stock_benchmarks_2000_2010_202...,2000/2010/2021/2024 total-housing-stock benchm...
3,Housing Production,powerbi_dof_annual_housing_stock_2020_2025.csv,"Annual 2020-2025 housing-stock totals, occupie..."


In [259]:
# ============================================================
# Final Workstream 1 completion checklist
# ============================================================

workstream1_completion_checklist = pd.DataFrame(
    [
        {
            "requirement": "RHNA allocation total and by income category",
            "status": "Complete",
            "output": "Main RHNA/production Power BI fact table",
        },
        {
            "requirement": (
                "RHNA progress by region, jurisdiction, and income category"
            ),
            "status": "Complete",
            "output": (
                "Allocation, qualifying units, remaining units, percent complete"
            ),
        },
        {
            "requirement": "Applications submitted",
            "status": "Complete",
            "output": "APR Table A, 2018-2025",
        },
        {
            "requirement": "Entitlements / planning approvals",
            "status": "Complete",
            "output": "APR Table A2, stage-date filtered",
        },
        {
            "requirement": "Building permits issued",
            "status": "Complete",
            "output": "APR Table A2, stage-date filtered",
        },
        {
            "requirement": "Units under construction",
            "status": "Documented unavailable",
            "output": (
                "Not inferred from APR; add only if a separate defensible "
                "regionwide source becomes available"
            ),
        },
        {
            "requirement": "Completed units / certificates of occupancy",
            "status": "Complete",
            "output": "APR Table A2, stage-date filtered",
        },
        {
            "requirement": "Historical APR from 2018 onward",
            "status": "Complete",
            "output": "2018-2025",
        },
        {
            "requirement": "Production by housing type",
            "status": "Complete",
            "output": (
                "SFD, SFA, 2-4 combined, 5+, ADU, MH; permits and completions"
            ),
        },
        {
            "requirement": "Completed housing since 2018 by year/jurisdiction/type",
            "status": "Complete",
            "output": "Housing-type Power BI fact table",
        },
        {
            "requirement": "Current overall housing stock",
            "status": "Complete",
            "output": "DOF E-5 jurisdiction + countywide",
        },
        {
            "requirement": (
                "Stock by occupied/vacant and structure categories"
            ),
            "status": "Complete",
            "output": "DOF E-5 annual housing-stock fact table",
        },
        {
            "requirement": (
                "Housing stock benchmarks: 2000, 2010, 2021, 2024"
            ),
            "status": "Complete",
            "output": "Census/ACS benchmark Power BI table",
        },
        {
            "requirement": "Annual housing supply estimates",
            "status": "Complete",
            "output": "DOF E-5 2020-2025",
        },
        {
            "requirement": "Housing deficit context",
            "status": "Complete - intentionally not calculated",
            "output": (
                "Existing stock, RHNA allocation, and remaining RHNA need "
                "are kept separate; no unsupported deficit measure is created"
            ),
        },
    ]
)

checklist_path = (
    DOCS_DIR / "workstream1_completion_checklist.csv"
)

workstream1_completion_checklist.to_csv(
    checklist_path,
    index=False,
)

print("Workstream 1 completion checklist:")
display(workstream1_completion_checklist)

blocking = workstream1_completion_checklist[
    ~workstream1_completion_checklist["status"].isin(
        [
            "Complete",
            "Documented unavailable",
            "Complete - intentionally not calculated",
        ]
    )
]

assert blocking.empty, (
    "One or more Workstream 1 requirements still need action."
)

print()
print(
    "Workstream 1 core data pipeline is complete for the specified "
    "requirements. The only non-populated stage is units under construction, "
    "which is explicitly documented as unavailable from the regionwide APR "
    "source rather than inferred."
)
print("Saved checklist:", checklist_path)


Workstream 1 completion checklist:


,requirement,status,output
0,RHNA allocation total and by income category,Complete,Main RHNA/production Power BI fact table
1,"RHNA progress by region, jurisdiction, and inc...",Complete,"Allocation, qualifying units, remaining units,..."
2,Applications submitted,Complete,"APR Table A, 2018-2025"
3,Entitlements / planning approvals,Complete,"APR Table A2, stage-date filtered"
4,Building permits issued,Complete,"APR Table A2, stage-date filtered"
5,Units under construction,Documented unavailable,Not inferred from APR; add only if a separate ...
6,Completed units / certificates of occupancy,Complete,"APR Table A2, stage-date filtered"
7,Historical APR from 2018 onward,Complete,2018-2025
8,Production by housing type,Complete,"SFD, SFA, 2-4 combined, 5+, ADU, MH; permits a..."
9,Completed housing since 2018 by year/jurisdict...,Complete,Housing-type Power BI fact table



Workstream 1 core data pipeline is complete for the specified requirements. The only non-populated stage is units under construction, which is explicitly documented as unavailable from the regionwide APR source rather than inferred.
Saved checklist: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/docs/workstream1_completion_checklist.csv
